# Exploratory Data Analysis of the ScreenSpot-Pro Windows Subset

**Name:** John-Ronan Beira  
**Date:** 19 November 2026  
**Dataset:** ScreenSpot-Pro (Li et al., 2025) — 1,020 grounding requests over 17 applications

> **How this notebook is organised.** The analysis functions live in four modules that the
> notebook writes out and imports, rather than being pasted into cells. `%%writefile` works
> in JupyterLab and in Google Colab, so the notebook still runs end to end on a machine that
> has nothing installed but the libraries in the setup cell — which is what makes it
> submittable. Every module is reproduced in full below, so nothing you are asked to trust is
> hidden: the cells are visible, runnable source. `src/isdm/eda/` in the repository holds the
> same four files, and a test asserts they are byte-identical.

## 1. Introduction

### What the data is

ScreenSpot-Pro is a GUI-grounding benchmark for high-resolution professional software. An
annotator was shown a full-screen screenshot of a real application running on Windows at the
machine's native resolution, plus a natural-language instruction such as *"Hand-draw cloud
line"*, and marked the single rectangular screen region that satisfies it. The screenshots
are native, uncropped captures of genuine installed software — AutoCAD Mechanical 2019,
Blender 4.0.2, SolidWorks Premium 2018, Unreal Engine 5.4.4, and others — not mockups.

The benchmark exists because high-resolution professional applications are the hard case for
GUI grounding: the interface is dense, the text is small, and a target can occupy a few
hundredths of a percent of the screen. The authors report that the best model at publication
reached 18.9%. Target geometry is worth measuring rather than assuming.

### Where it came from

| Item | Value |
|---|---|
| Source | Hugging Face dataset `likaixin/ScreenSpot-Pro`, revision `main` |
| Link | https://huggingface.co/datasets/likaixin/ScreenSpot-Pro |
| Paper | Li et al. (2025), arXiv:2504.07981 |
| Licence | MIT (dataset card); the screenshots remain subject to the application vendors' rights |
| Files used | 16 `annotations/*_windows.json` files plus `annotations/matlab_macos.json`, and `images/*.png` (1,020 screenshots) |
| Retrieved | 19 November 2026 |
| Local path | `data/screenspot_pro_windows/` |
| Volume | 1,020 requests, 1,020 screenshots, 862.1 MB of PNG |

The 17 per-application annotation files were merged into `grounding_requests.csv` with one
row per request, and every screenshot the annotations reference was downloaded. The merge is
verified against the raw files in section 2, and one platform label contradicts its own
captures (section 3).

### Data dictionary

| Column | Meaning | Unit |
|---|---|---|
| `id` | Request identifier, `<application>_<platform>_<n>` (MATLAB rows use `matlab_macos_<n>`) | label |
| `application` | Application the screenshot is taken from | label (17 levels) |
| `group` | Industry category assigned by the benchmark | label (6 levels) |
| `platform` | Platform label in the release; two levels (`windows` 927, `macos` 93) | label |
| `ui_type` | Whether the annotated target is an **icon** or a **text** element | label (2 levels) |
| `instruction` | The English grounding instruction | free text |
| `instruction_cn` | Chinese translation supplied by the dataset | free text |
| `img_filename` | Screenshot path relative to `images/` | label |
| `img_size_w`, `img_size_h` | Screenshot width and height as declared in the annotation | pixels |
| `bbox_x1`, `bbox_y1` | Target box top-left corner, from the top-left screen origin | pixels |
| `bbox_x2`, `bbox_y2` | Target box bottom-right corner | pixels |

The release labels MATLAB as `macos` in `platform`, in its `id` (`matlab_macos_<n>`) and in its
image directory (`matlab_mac/`). The captures themselves are Windows: for example
`matlab_mac/screenshot_2024-11-30_20-47-44.png` shows a Windows 10 taskbar, right-hand window
controls and a `D:\annotation` working directory. Section 4 corrects the `platform` label; the
identifier and image path keep the release's spelling.

### What one row represents

**One row is one grounding request**: one screenshot, one instruction, one target box. The
corpus is not a sample of screens in the strict sense, since 1,020 rows carry 1,020 distinct
filenames, and it is not a sample of elements either: only the requested element is annotated.
Section 3 shows that 14 pairs of those filenames carry byte-identical images, so 1,020 requests
describe 992 distinct screen contents.

### The questions this analysis answers

1. **Q1 — Target size.** How large is the annotated target relative to its screen, and does
   that differ between icon targets and text targets?
2. **Q2 — Corpus balance.** How evenly are the 1,020 requests spread across the 17 applications
   and 6 industry groups, and what does the imbalance permit?
3. **Q3 — Resolution and size.** Is screen resolution associated with relative target size,
   and can that association be separated from the application each screen comes from?
4. **Q4 — Instruction text.** How long and how repetitive are the instructions, and does that
   vary by target type?

All four are answerable from the annotations alone: no model is run, so nothing here depends
on results that do not yet exist.

## 2. Data loading and first look

The assignment specifies this setup cell. It imports the analysis stack, widens the column
display, sets a plotting theme, and fixes the seed used by every sample in this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42  # use this anywhere you sample

That cell defines the stack but no paths. A notebook kernel's working directory is the
notebook's own folder, not the repository root, so a path written as `data/...` would
resolve differently depending on how Jupyter or `nbconvert` was launched. The next cell
locates the data explicitly and prepares the directory the analysis modules are written into.

`ROOT` is found by walking up from the working directory until a folder containing
`data/screenspot_pro_windows` appears, so the same notebook runs from the repository, from
`notebooks/`, or from a Colab session with the data uploaded beside it. No path on this
machine is written into the notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

DATASET_FOLDER = Path("data") / "screenspot_pro_windows"
ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / DATASET_FOLDER).is_dir()),
    Path.cwd(),
)
DATA_DIR = ROOT / DATASET_FOLDER

MODULE_DIR = Path("eda_modules")
MODULE_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(MODULE_DIR.resolve()))

print("working directory :", Path.cwd())
print("dataset directory :", DATA_DIR)
print("module directory  :", MODULE_DIR.resolve())

The dataset folder was found and the module directory is ready. Nothing is read yet.

The next four cells write the analysis modules. They are ordinary source files — docstrings,
type hints, no notebook context — and each cell prints `Overwriting ...` when it runs. Reading
them is the honest way to check what the later cells actually do.

`eda_quality` measures what is wrong with the data: completeness, duplication, bounding-box
validity, outliers, resolution support, and the integrity of the screenshot files.

In [ ]:
%%writefile eda_modules/eda_quality.py
"""Measure data quality: completeness, duplication, validity, and screenshot integrity.

Why this module exists: every claim in an exploratory analysis rests on a measurement of
what is wrong with the data, and those measurements are mechanical. Keeping them here
means the notebook shows *what was measured and what it concluded*, while the counting
logic is importable and testable on its own.

All functions take a DataFrame and return a DataFrame or a plain tuple; none of them plot
or print, so a caller decides how results are displayed.
"""

from __future__ import annotations

import json
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd

# Columns that identify a request rather than measure it.
ID_COLUMNS = ("id", "img_filename")


def requests_vs_annotations(frame: pd.DataFrame, annotation_dir: Path) -> pd.DataFrame:
    """Compare the merged request table against the raw per-application JSON files.

    Args:
        frame: the loaded request table.
        annotation_dir: directory holding ``*_windows.json`` annotation files.

    Returns:
        One row per annotation file with its record count, plus a ``merged`` column
        marking whether that count is measured against the merged table.

    The merge step that produced the table is the one place rows can silently vanish, so
    this is checked before any analysis rather than assumed. A shortfall here invalidates
    every later number, which is why the function reports the difference instead of only
    an equality flag.
    """
    # 1. Count the records each raw file declares, and how many of its ids reached the
    #    merged table. Membership is tested by id rather than inferred from the filename,
    #    because a file's name is not evidence of which requests it contributes — the
    #    release files MATLAB as `matlab_macos.json` while its records are Windows.
    known_ids = set(frame["id"])
    per_file = []
    for path in sorted(annotation_dir.glob("*.json")):
        records = json.loads(path.read_text())
        if not isinstance(records, list):
            continue
        identifiers = [record["id"] for record in records]
        per_file.append(
            {
                "file": path.name,
                "records": len(identifiers),
                "in_merged_csv": sum(1 for identifier in identifiers if identifier in known_ids),
            }
        )

    table = pd.DataFrame(per_file).sort_values("records", ascending=False)

    # 2. Attach the merged total to every row so a caller can print both numbers side by side.
    table["merged_total"] = len(frame)

    # 3. A file whose raw count differs from its contribution to the merged table means
    #    the merge either dropped records or duplicated them.
    table["difference"] = table["records"] - table["in_merged_csv"]
    return table


def missing_table(frame: pd.DataFrame) -> pd.DataFrame:
    """Count and percentage of missing values per column, worst first.

    Args:
        frame: the request table.

    Returns:
        One row per column with ``n_missing`` and ``pct`` columns.

    Percentages matter more than counts at this size: one missing cell in 927 rows is
    0.1%, which is easy to mistake for "no missing data" if only counts are read.
    """
    # 1. Count non-null absences per column.
    table = frame.isna().sum().to_frame("n_missing")
    # 2. Express them as a share of the row count so columns are comparable.
    table["pct"] = (table["n_missing"] / len(frame) * 100).round(3)
    # 3. Sort so the worst column, if any, is read first.
    return table.sort_values("n_missing", ascending=False)


def duplicate_report(frame: pd.DataFrame, text_column: str = "instruction") -> pd.Series:
    """Count exact duplicates of rows, identifiers, and instruction strings.

    Args:
        frame: the request table.
        text_column: the free-text column whose repetition is of interest.

    Returns:
        A named Series of counts.

    The three checks answer different questions. Duplicate rows mean a record was entered
    twice; duplicate ids mean the same request appears twice under one name; repeated
    instruction text is neither — it is a property of the task, recorded here because it
    limits how far the text can disambiguate a target.
    """
    return pd.Series(
        {
            "exact duplicate rows": int(frame.duplicated().sum()),
            "duplicate ids": int(frame["id"].duplicated().sum()),
            "duplicate (screenshot, instruction) pairs": int(
                frame.duplicated(subset=["img_filename", text_column]).sum()
            ),
            "rows sharing an instruction string": int(
                frame[text_column].duplicated(keep=False).sum()
            ),
            "distinct instruction strings reused": int(
                frame.loc[frame[text_column].duplicated(keep=False), text_column].nunique()
            ),
        },
        name="count",
    )


def level_audit(frame: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """List the levels of each categorical column and flag whitespace defects.

    Args:
        frame: the request table.
        columns: categorical columns to audit.

    Returns:
        One row per column with level count, sorted levels, and whether any level carries
        leading or trailing whitespace.

    Two spellings of one category split a group silently: every later share, median and
    test then runs on half the intended rows. The check is cheap, so it runs before any
    grouping rather than after a suspicious result.
    """
    rows = []
    for column in columns:
        values = frame[column].dropna().unique()
        rows.append(
            {
                "column": column,
                "n_levels": len(values),
                "levels": ", ".join(sorted(map(str, values))),
                "no_whitespace": all(str(value) == str(value).strip() for value in values),
            }
        )
    return pd.DataFrame(rows)


def bbox_violations(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Check bounding boxes for impossible or out-of-screen geometry.

    Args:
        frame: table with ``bbox_x1``/``bbox_y1``/``bbox_x2``/``bbox_y2`` and
            ``img_size_w``/``img_size_h``.

    Returns:
        A tuple of (counts per violation type, the offending rows).

    Four independent failures are counted because they have different causes: inverted
    corners mean a coordinate convention error, zero area means an empty label, and an
    out-of-bounds corner means either a wrong coordinate space or an off-by-one at the
    screen edge. Collapsing them into one "invalid boxes" number would hide which repair
    is needed.
    """
    x1, y1 = frame["bbox_x1"], frame["bbox_y1"]
    x2, y2 = frame["bbox_x2"], frame["bbox_y2"]
    width, height = frame["img_size_w"], frame["img_size_h"]

    counts = pd.Series(
        {
            "x2 <= x1 (inverted horizontally)": int((x2 <= x1).sum()),
            "y2 <= y1 (inverted vertically)": int((y2 <= y1).sum()),
            "zero area": int(((x2 - x1) * (y2 - y1) == 0).sum()),
            "x1 < 0": int((x1 < 0).sum()),
            "y1 < 0": int((y1 < 0).sum()),
            "x2 > image width": int((x2 > width).sum()),
            "y2 > image height": int((y2 > height).sum()),
        },
        name="rows affected",
    )

    offending = frame[(x1 < 0) | (y1 < 0) | (x2 > width) | (y2 > height)]
    return counts.to_frame(), offending


def iqr_flags(series: pd.Series, multiplier: float = 1.5) -> tuple[pd.Series, float, float]:
    """Flag values outside ``[Q1 - k*IQR, Q3 + k*IQR]``.

    Args:
        series: values to flag.
        multiplier: fence width in IQRs; 1.5 is the conventional choice.

    Returns:
        A tuple of (boolean mask aligned to ``series``, lower fence, upper fence).

    Flagging is deliberately separate from removing. The mask says "unusual", and only a
    human decision can say "erroneous"; deleting on the flag alone removes exactly the
    real extreme values an analysis of difficulty most needs to keep.
    """
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - multiplier * iqr, q3 + multiplier * iqr
    return (series < lower) | (series > upper), float(lower), float(upper)


def outlier_summary(frame: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Apply the IQR rule to several columns and summarise the result.

    Args:
        frame: the request table.
        columns: numeric columns to test.

    Returns:
        One row per column with the flag count, share of rows, and both fences.
    """
    rows = []
    for column in columns:
        mask, lower, upper = iqr_flags(frame[column])
        rows.append(
            {
                "column": column,
                "n_flagged": int(mask.sum()),
                "pct_flagged": round(mask.mean() * 100, 2),
                "lower_fence": round(lower, 4),
                "upper_fence": round(upper, 4),
            }
        )
    return pd.DataFrame(rows)


def resolution_support(frame: pd.DataFrame) -> pd.DataFrame:
    """Count requests per native screen resolution.

    Args:
        frame: the request table.

    Returns:
        One row per resolution with its request count and share.

    A resolution carried by a handful of rows cannot support a per-resolution claim, so
    the count is the support for any later statement and is read before making one.
    """
    support = (
        frame.groupby(["img_size_w", "img_size_h"])
        .size()
        .rename("requests")
        .to_frame()
    )
    support["share_pct"] = (support["requests"] / len(frame) * 100).round(2)
    return support.sort_index()


def audit_images(frame: pd.DataFrame, image_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Open every referenced screenshot and compare it with its annotation.

    Args:
        frame: the request table.
        image_root: directory that ``img_filename`` values are relative to.

    Returns:
        A tuple of (one audit row per distinct screenshot, the request table joined to it).

    A declared ``img_size`` is not evidence of anything. Opening the file establishes
    three facts at once: the file exists, it decodes, and its pixel dimensions match the
    coordinate space the bounding boxes were written in. If dimensions disagreed, every
    size measure in the analysis would be measured against the wrong denominator.
    """
    records = []
    for relative in sorted(frame["img_filename"].unique()):
        path = image_root / relative
        if not path.exists():
            records.append((relative, None, None, None, "missing"))
            continue
        try:
            # PIL reads the header for size and mode, so no full decode is needed here.
            from PIL import Image

            with Image.open(path) as image:
                records.append((relative, image.width, image.height, image.mode, "ok"))
        except Exception as error:  # noqa: BLE001 - report the failure, keep auditing
            records.append((relative, None, None, None, f"unreadable: {type(error).__name__}"))

    audit = pd.DataFrame(
        records, columns=["img_filename", "file_w", "file_h", "mode", "status"]
    )
    joined = frame.merge(audit, on="img_filename", how="left")
    return audit, joined


def size_mismatch(joined: pd.DataFrame) -> pd.DataFrame:
    """Return rows whose real pixel size disagrees with the animated ``img_size``.

    Args:
        joined: output of :func:`audit_images`.

    Returns:
        The offending rows, empty when every file matches its annotation.
    """
    width_differs = joined["file_w"] != joined["img_size_w"]
    height_differs = joined["file_h"] != joined["img_size_h"]
    return joined[width_differs | height_differs]


def average_hash(path: Path, size: int = 8) -> int:
    """Return a 64-bit average hash of an image.

    Args:
        path: image file.
        size: side length of the reduced greyscale thumbnail; 8 yields 64 bits.

    Returns:
        A ``numpy.uint64`` whose bits say whether each thumbnail pixel exceeded the mean.

    The unsigned 64-bit type is load-bearing: a 64-bit hash overflows signed ``int64``, and
    an object-dtype array of Python integers cannot be shifted as a block, which is how the
    pairwise comparison below is vectorised.

    The transform is deliberate: downscaling to 8x8 removes detail, converting to
    greyscale removes colour, and thresholding at the mean removes overall brightness.
    What survives is coarse layout, which is what makes a re-saved or re-captured screen
    hash close to the original while a different screen does not.
    """
    from PIL import Image

    with Image.open(path) as image:
        thumbnail = image.convert("L").resize((size, size), Image.Resampling.LANCZOS)
    pixels = np.asarray(thumbnail, dtype=float).ravel()
    bits = pixels > pixels.mean()

    # Pack the 64 booleans into one integer, most significant bit first.
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return np.uint64(value)


def file_sha256(path: Path) -> str:
    """Return the SHA-256 hex digest of a file's bytes.

    Args:
        path: file to digest.

    Returns:
        Lower-case hex digest.

    Read in chunks rather than whole: the largest screenshots are a few megabytes each and
    there are 927 of them, so streaming keeps the memory cost flat.
    """
    import hashlib

    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(1 << 20):
            digest.update(chunk)
    return digest.hexdigest()


def byte_duplicates(files: Iterable[str], image_root: Path) -> pd.DataFrame:
    """Find files whose bytes are identical.

    Args:
        files: screenshot names relative to ``image_root``.
        image_root: directory holding the screenshots.

    Returns:
        One row per file that shares its digest with another, with the digest and the size
        of its duplicate group; empty when no two files are byte-identical.

    This is the decisive duplicate check. A digest cannot be fooled by two screens looking
    alike — equal digests mean the same file — so it is the check that establishes whether
    the corpus is free of re-saves, while the perceptual hash is only suggestive.
    """
    records = [(name, file_sha256(image_root / name)) for name in files]
    table = pd.DataFrame(records, columns=["img_filename", "sha256"])
    group_sizes = table["sha256"].map(table["sha256"].value_counts())
    duplicated = table[group_sizes > 1].copy()
    duplicated["group_size"] = group_sizes[group_sizes > 1]
    return duplicated.sort_values(["sha256", "img_filename"]).reset_index(drop=True)


def hash_distance_summary(
    files: Iterable[str], hashes: np.ndarray, thresholds: Iterable[int] = (0, 1, 2, 4, 8, 16)
) -> pd.DataFrame:
    """Count screenshot pairs whose perceptual hashes fall within each Hamming threshold.

    Args:
        files: screenshot names, aligned with ``hashes``.
        hashes: average hashes from :func:`average_hash`.
        thresholds: Hamming distances to count pairs at.

    Returns:
        One row per threshold with the pair count, the same-application count, and the
        share of all pairs.

    The same-application column is the diagnostic that matters. An average hash of a
    downscaled screenshot mostly encodes window chrome and panel layout, which every
    capture of one application shares; if all close pairs are within a single application
    and none cross applications, the hash is measuring layout rather than content and
    cannot be used to claim two screens are near-duplicates.
    """
    files = list(files)
    packed = np.asarray(hashes, dtype=np.uint64)
    shifts = np.arange(63, -1, -1, dtype=np.uint64)
    bit_planes = ((packed[:, None] >> shifts) & 1).astype(np.int8)
    distance = (bit_planes[:, None, :] != bit_planes[None, :, :]).sum(axis=2)

    rows, columns = np.triu_indices(len(files), k=1)
    values = distance[rows, columns]
    names = np.asarray(files)
    same_application = np.array(
        [
            names[i].split("/")[0] == names[j].split("/")[0]
            for i, j in zip(rows, columns, strict=True)
        ]
    )

    total_pairs = len(values)
    summary = []
    for threshold in thresholds:
        within = values <= threshold
        summary.append(
            {
                "max_distance": threshold,
                "pairs": int(within.sum()),
                "same_application": int((within & same_application).sum()),
                "cross_application": int((within & ~same_application).sum()),
                "pct_of_all_pairs": round(within.mean() * 100, 3),
                "total_pairs": total_pairs,
            }
        )
    return pd.DataFrame(summary)

`eda_features` derives the analysis columns — relative target size, position, resolution,
text length — in one place so "target size" means the same thing in every section. It also
applies the repairs and returns the cleaning log.

In [ ]:
%%writefile eda_modules/eda_features.py
"""Derive analysis features and apply the documented cleaning steps.

Why this module exists: "target size" has to mean one thing throughout the analysis. If
each section recomputed it inline, one section could divide by screen area and another by
screen width and the numbers would silently disagree. Deriving the features in one
function makes that impossible.

Cleaning lives beside it because the derived features depend on the cleaned coordinates,
and the log returned with the cleaned frame is the audit trail the report renders.
"""

from __future__ import annotations

import numpy as np
import pandas as pd


def add_derived_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of ``frame`` with the derived analysis columns added.

    Args:
        frame: table with ``bbox_x1``/``bbox_y1``/``bbox_x2``/``bbox_y2``,
            ``img_size_w``/``img_size_h`` and an ``instruction`` column.

    Returns:
        A new DataFrame with geometry, position, resolution and text-length features.

    The relative measures are the point. Absolute pixel area is not comparable between a
    1920x1080 capture and a 5120x1440 capture, so ``target_area_pct`` — the box's share
    of screen area — is the difficulty measure the analysis uses, and the pixel columns
    are kept only as the intermediate values that produce it.
    """
    derived = frame.copy()

    # 1. Box geometry from its corners.
    derived["bbox_w"] = derived["bbox_x2"] - derived["bbox_x1"]
    derived["bbox_h"] = derived["bbox_y2"] - derived["bbox_y1"]
    derived["bbox_area_px"] = derived["bbox_w"] * derived["bbox_h"]
    derived["screen_area_px"] = derived["img_size_w"] * derived["img_size_h"]

    # 2. Relative size. Percent of screen area is the headline measure; the per-axis
    #    shares say whether a target is wide-and-thin or compact.
    derived["target_area_pct"] = derived["bbox_area_px"] / derived["screen_area_px"] * 100
    derived["bbox_w_pct"] = derived["bbox_w"] / derived["img_size_w"] * 100
    derived["bbox_h_pct"] = derived["bbox_h"] / derived["img_size_h"] * 100

    # 3. Screen shape and scale, so resolution effects can be examined separately from
    #    target shape.
    derived["screen_megapixels"] = derived["screen_area_px"] / 1e6
    derived["screen_aspect"] = derived["img_size_w"] / derived["img_size_h"]
    derived["bbox_aspect"] = derived["bbox_w"] / derived["bbox_h"]

    # 4. Where on the screen the target sits, as 0-1 fractions of each axis.
    derived["bbox_cx_rel"] = (derived["bbox_x1"] + derived["bbox_w"] / 2) / derived["img_size_w"]
    derived["bbox_cy_rel"] = (derived["bbox_y1"] + derived["bbox_h"] / 2) / derived["img_size_h"]

    # 5. Distance to the nearest screen edge, scaled by the shorter side so a 4:3 and a
    #    32:9 screen stay comparable. Zero means the target touches a screen boundary.
    edge_gaps = np.column_stack(
        [
            derived["bbox_x1"],
            derived["bbox_y1"],
            derived["img_size_w"] - derived["bbox_x2"],
            derived["img_size_h"] - derived["bbox_y2"],
        ]
    ).min(axis=1)
    derived["edge_distance_pct"] = (
        edge_gaps / derived[["img_size_w", "img_size_h"]].min(axis=1) * 100
    )

    # 6. Instruction length in both units; words are the unit the analysis compares,
    #    characters indicate how long the individual tokens are.
    derived["instruction_chars"] = derived["instruction"].str.len()
    derived["instruction_words"] = derived["instruction"].str.split().str.len()

    return derived


DERIVED_COLUMNS = (
    "target_area_pct",
    "bbox_w_pct",
    "bbox_h_pct",
    "bbox_aspect",
    "screen_megapixels",
    "screen_aspect",
    "bbox_cx_rel",
    "bbox_cy_rel",
    "edge_distance_pct",
    "instruction_chars",
    "instruction_words",
    "bbox_area_px",
    "screen_area_px",
)


def clean_requests(
    frame: pd.DataFrame, derived: pd.DataFrame | None = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply the documented repairs and return the cleaned table with its change log.

    Args:
        frame: the raw request table. Left unmodified.
        derived: optionally a frame already passed through :func:`add_derived_features`;
            when omitted the features are computed here so the returned frame is always
            analysis-ready.

    Returns:
        A tuple of (cleaned frame, cleaning log). The log has one row per change with the
        problem, the action, what it touched, how many rows, and why — and one row
        recording the checks that found nothing, so an empty log cannot be confused with
        a skipped audit.

    Two rules shape the implementation. No row is dropped for a fixable defect: dropping
    removes evidence to repair a coordinate. And every change is counted before it is
    made, so the log states a measurement rather than an impression.
    """
    cleaned = (derived if derived is not None else add_derived_features(frame)).copy()
    entries: list[dict[str, object]] = []

    def record(problem, action, affected, n_rows, reason):
        """Append one log line; a nested helper keeps the call sites readable."""
        entries.append(
            {
                "#": len(entries) + 1,
                "problem": problem,
                "action": action,
                "columns/rows affected": affected,
                "n_rows": n_rows,
                "reason": reason,
            }
        )

    # Change 1: instruction text carries stray whitespace. Strip it and collapse runs of
    # spaces, because the text-length features are derived from this string and a trailing
    # space inflates a character count by one.
    normalized = cleaned["instruction"].str.strip().str.replace(r"\s{2,}", " ", regex=True)
    whitespace_changed = normalized != cleaned["instruction"]
    if whitespace_changed.any():
        affected = int(whitespace_changed.sum())
        cleaned["instruction"] = normalized
        record(
            "instruction text with leading/trailing spaces or doubled spaces",
            "stripped and collapsed whitespace",
            "instruction, 1 column",
            affected,
            "character counts are derived from this string; whitespace is not content",
        )

    # Change 2: a platform label contradicted by the captures themselves. The release
    # files MATLAB as `macos` in the record, the filename and the image directory, and
    # ships no `matlab_windows.json`. Sampled screenshots show a Windows 10 taskbar,
    # right-hand window controls and a `D:\annotation` working directory, and the paper's
    # application table lists MATLAB as Windows, so the label is wrong for these rows.
    mislabelled = cleaned["application"].eq("matlab") & cleaned["platform"].ne("windows")
    if mislabelled.any():
        affected = int(mislabelled.sum())
        cleaned.loc[mislabelled, "platform"] = "windows"
        record(
            f"{affected} MATLAB records labelled platform=macos "
            "(also filename and image directory)",
            "corrected platform to windows",
            "platform, 1 column",
            affected,
            "sampled captures show a Windows 10 taskbar and a D:\\ path; the paper lists MATLAB "
            "as Windows",
        )

    # Change 3: a coordinate outside the screen rectangle. Clamp it to the first valid
    # pixel; the target itself is unambiguous, only the corner is off by one.
    off_screen = cleaned["bbox_y1"] < 0
    if off_screen.any():
        affected = int(off_screen.sum())
        identifiers = ", ".join(cleaned.loc[off_screen, "id"].head(3))
        cleaned.loc[off_screen, "bbox_y1"] = 0
        record(
            f"bbox_y1 negative on {identifiers} (outside the screen)",
            "clamped bbox_y1 to 0",
            "bbox_y1, 1 column",
            affected,
            "minimum valid screen coordinate is 0; target sits flush to the top edge",
        )

    # Nothing else was found: record that explicitly rather than leaving the log with a
    # single line and no statement about the checks that passed.
    record(
        "no missing values, duplicate rows, duplicate ids, inverted or zero-area boxes",
        "no action",
        "none",
        0,
        "measured in the quality assessment; nothing to repair",
    )

    # Any coordinate repaired above changes the derived geometry, so recompute the
    # features rather than leaving a stale width or area beside a corrected corner.
    cleaned = add_derived_features(cleaned.drop(columns=list(DERIVED_COLUMNS), errors="ignore"))

    return cleaned, pd.DataFrame(entries)


def variance_share(frame: pd.DataFrame, value: str, groups: str) -> dict[str, float]:
    """Decompose the variance of ``value`` into a between-group and within-group part.

    Args:
        frame: table containing ``value`` and ``groups``.
        value: numeric column to decompose.
        groups: grouping column.

    Returns:
        ``{"between_pct": ..., "within_pct": ..., "n_groups": ...}``.

    The decomposition answers whether rows inside one group are more alike than rows
    across groups. That decides whether a group-level effect is a real grouping factor or
    just an artefact of pooling. It is computed on the log of the value because a
    right-skewed measure would otherwise have its variance dominated by its largest
    observations, making the between-group share meaningless.
    """
    # 1. Work on a log scale; zero or negative values cannot be logged and indicate a
    #    caller error rather than something to silently drop.
    values = frame[value]
    if (values <= 0).any():
        raise ValueError(f"{value} must be strictly positive to decompose on a log scale")
    logged = np.log10(values)

    # 2. Total sum of squares around the grand mean.
    grand_mean = logged.mean()
    total = ((logged - grand_mean) ** 2).sum()

    # 3. Between-group sum of squares: each group's mean distance from the grand mean,
    #    weighted by its size.
    grouped = logged.groupby(frame[groups])
    between = float(((grouped.mean() - grand_mean) ** 2 * grouped.size()).sum())

    return {
        "between_pct": round(between / total * 100, 2),
        "within_pct": round((1 - between / total) * 100, 2),
        "n_groups": int(frame[groups].nunique()),
    }

`eda_analysis` holds the summaries and tests: distributions, rank comparisons, correlations,
the subgroup check, crosstabs, and the text measures. Nothing in it prints, so the notebook
owns the interpretation.

In [ ]:
%%writefile eda_modules/eda_analysis.py
"""Compute the summaries and tests the report quotes.

Why this module exists: the analysis repeatedly asks the same three questions — how is
this variable distributed, do these two groups differ, and is this association real or a
composition effect. Each answer is one function, so the notebook can state *which*
question it is answering in Markdown and get a consistent, testable answer underneath.

Every function returns a table or a plain mapping. Nothing prints, so the caller owns the
presentation and the interpretation stays in the report.
"""

from __future__ import annotations

import re
from collections import Counter
from collections.abc import Iterable

import numpy as np
import pandas as pd
from scipy import stats


def numeric_summary(frame: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    """Summarise numeric columns with centre, spread and shape.

    Args:
        frame: the cleaned table.
        columns: numeric columns to summarise.

    Returns:
        One row per column with count, mean, median, standard deviation, IQR, min, max
        and skewness.

    The mean is included deliberately even where it is a poor summary, because the
    contrast with the median is the evidence that the variable is skewed and that the
    median is the right one to report. IQR is appended beside the standard deviation for
    the same reason: it survives the outliers that the standard deviation does not.
    """
    columns = list(columns)
    summary = frame[columns].describe().T
    summary["skew"] = frame[columns].skew()
    summary["IQR"] = frame[columns].quantile(0.75) - frame[columns].quantile(0.25)
    ordered = ["count", "mean", "50%", "std", "IQR", "min", "max", "skew"]
    return summary[ordered].round(4)


def rank_comparison(left: pd.Series, right: pd.Series, left_label: str, right_label: str) -> dict:
    """Compare two groups on a skewed measure with a Mann-Whitney U test.

    Args:
        left: values for the first group.
        right: values for the second group.
        left_label: display name of the first group.
        right_label: display name of the second group.

    Returns:
        A mapping of group sizes, medians, IQRs, the U statistic, its p value, the ratio
        of medians, and the rank-biserial effect size.

    A rank test rather than a t-test because target area spans three orders of magnitude
    and is strongly skewed; the median ratio is reported beside the p value because a
    significant difference on 927 rows can be trivially small in practice, and the ratio
    is what says whether the gap matters.
    """
    statistic, p_value = stats.mannwhitneyu(left, right, alternative="two-sided")

    # Rank-biserial correlation, aligned with the U statistic: U counts pairs where the
    # left group exceeds the right, so r = +1 means every left value is *below* every
    # right value and r = 0 means the groups fully interleave.
    effect = 1 - 2 * statistic / (len(left) * len(right))

    return {
        "left_label": left_label,
        "right_label": right_label,
        "left_n": len(left),
        "right_n": len(right),
        "left_median": round(float(left.median()), 6),
        "right_median": round(float(right.median()), 6),
        "left_iqr": (
            round(float(left.quantile(0.25)), 6),
            round(float(left.quantile(0.75)), 6),
        ),
        "right_iqr": (
            round(float(right.quantile(0.25)), 6),
            round(float(right.quantile(0.75)), 6),
        ),
        "u_statistic": float(statistic),
        "p_value": float(p_value),
        "median_ratio": round(float(right.median() / left.median()), 3),
        "rank_biserial": round(float(effect), 3),
    }


def spearman_summary(frame: pd.DataFrame, x: str, y: str) -> dict:
    """Report the Spearman rank correlation between two columns.

    Args:
        frame: the cleaned table.
        x: first column.
        y: second column.

    Returns:
        A mapping with the correlation, its p value, and the number of rows used.

    Spearman rather than Pearson because neither variable is normal and the relation need
    not be linear. The p value is reported but not treated as proof: at this sample size
    almost any non-zero correlation is significant, which is why the correlation itself
    and its subgroup stability carry the argument.
    """
    usable = frame[[x, y]].dropna()
    rho, p_value = stats.spearmanr(usable[x], usable[y])
    return {
        "x": x,
        "y": y,
        "rho": round(float(rho), 3),
        "p_value": float(p_value),
        "n": len(usable),
    }


def subgroup_correlations(
    frame: pd.DataFrame,
    group: str,
    x: str,
    y: str,
    min_rows: int = 20,
    min_levels: int = 3,
) -> pd.DataFrame:
    """Recompute a correlation inside each group to expose a composition effect.

    Args:
        frame: the cleaned table.
        group: grouping column.
        x: first correlated column.
        y: second correlated column.
        min_rows: groups smaller than this are reported as unusable, not tested.
        min_levels: groups whose ``x`` takes fewer distinct values than this cannot
            support a rank correlation and are skipped.

    Returns:
        One row per group with its size, distinct x values, correlation and p value, plus
        a note when the group could not be tested.

    A pooled correlation between two variables that are both determined by group
    membership is not evidence about the variables: it can appear, disappear or reverse
    once group is held constant. Recomputing inside each group is the check, and the
    groups that cannot support one are named rather than quietly dropped.
    """
    rows = []
    for name, chunk in frame.groupby(group):
        distinct = int(chunk[x].nunique())
        if len(chunk) < min_rows or distinct < min_levels:
            rows.append(
                {
                    "group": name,
                    "n": len(chunk),
                    "distinct_x": distinct,
                    "rho": np.nan,
                    "p_value": np.nan,
                    "note": "insufficient size or variation",
                }
            )
            continue
        rho, p_value = stats.spearmanr(chunk[x], chunk[y])
        rows.append(
            {
                "group": name,
                "n": len(chunk),
                "distinct_x": distinct,
                "rho": round(float(rho), 3),
                "p_value": float(p_value),
                "note": "",
            }
        )
    return pd.DataFrame(rows).sort_values("rho")


def crosstab_share(frame: pd.DataFrame, index: str, columns: str) -> tuple[pd.DataFrame, dict]:
    """Build a row-percentage crosstab and test the two variables for independence.

    Args:
        frame: the cleaned table.
        index: row variable.
        columns: column variable.

    Returns:
        A tuple of (row-share percentages with a size column, chi-square result).

    Row percentages rather than raw counts because the row groups differ in size by a
    factor of more than two: raw counts would present the largest group's composition as
    if it were the overall pattern. The chi-square result is returned rather than printed
    so the caller decides whether the test is worth quoting.
    """
    counts = pd.crosstab(frame[index], frame[columns])
    shares = (counts.div(counts.sum(axis=1), axis=0) * 100).round(2)
    shares["n"] = counts.sum(axis=1)

    chi2, p_value, dof, expected = stats.chi2_contingency(counts)
    return shares, {
        "chi2": round(float(chi2), 2),
        "dof": int(dof),
        "p_value": float(p_value),
        "min_expected": round(float(expected.min()), 2),
    }


def application_medians(
    frame: pd.DataFrame, value: str, group: str = "application"
) -> pd.DataFrame:
    """Median and IQR of ``value`` per group, ordered from smallest to largest.

    Args:
        frame: the cleaned table.
        value: numeric column to summarise.
        group: grouping column.

    Returns:
        One row per group with its size, median and quartiles.

    Ordered by median because the comparison being made in the report is "how many times
    larger is the largest group's typical target than the smallest".
    """
    medians = (
        frame.groupby(group)[value]
        .agg(
            n="size",
            median="median",
            q1=lambda series: series.quantile(0.25),
            q3=lambda series: series.quantile(0.75),
        )
        .sort_values("median")
        .round(6)
    )
    return medians


def instruction_summary(frame: pd.DataFrame, text_column: str = "instruction") -> pd.DataFrame:
    """Describe instruction length in words and characters.

    Args:
        frame: the cleaned table.
        text_column: column holding the instruction text.

    Returns:
        One row of length statistics, plus the count of single-word instructions.

    Single-word instructions are counted separately because they are the extreme case for
    disambiguation: a bare "Copy" names a target type and nothing else.
    """
    words = frame["instruction_words"]
    characters = frame["instruction_chars"]
    return pd.DataFrame(
        {
            "words": [words.median(), words.mean(), words.min(), words.max()],
            "characters": [
                characters.median(),
                characters.mean(),
                characters.min(),
                characters.max(),
            ],
        },
        index=["median", "mean", "min", "max"],
    ).round(2).assign(single_word_instructions=int((words == 1).sum()))


def top_terms(
    series: pd.Series,
    stopwords: Iterable[str] = (),
    top_n: int = 20,
    min_length: int = 3,
) -> pd.DataFrame:
    """Count the most frequent content words after stop-word removal.

    Args:
        series: instruction strings.
        stopwords: words to exclude.
        top_n: how many terms to return.
        min_length: shortest token length to keep.

    Returns:
        A DataFrame of ``term`` and ``count``, most frequent first.

    Tokens are lower-cased and reduced to letters and hyphens, because the question is
    which interface concepts the instructions name — not how they were capitalised. A
    stop-word list supplied by the caller keeps this module free of a language assumption
    the data may not meet.
    """
    stopwords = {word.lower() for word in stopwords}
    counter: Counter[str] = Counter()

    for text in series.dropna():
        for token in re.findall(r"[a-z][a-z\-]+", str(text).lower()):
            if token not in stopwords and len(token) >= min_length:
                counter[token] += 1

    return pd.DataFrame(counter.most_common(top_n), columns=["term", "count"])


def position_usage(series: pd.Series, terms: Iterable[str]) -> pd.Series:
    """Mark instructions that name a position or direction.

    Args:
        series: instruction strings.
        terms: spatial terms to look for, matched case-insensitively.

    Returns:
        A boolean Series aligned to ``series``.

    An instruction that says "top left" has already narrowed the search, so a corpus with
    many of them would flatter any model tested on it. Counting them separates genuine
    localisation difficulty from textual spoon-feeding.

    Matching is on word boundaries, which is not cosmetic: a plain substring search counts
    "centerline" (contains "center"), "stop" (contains "top"), "Linear" / "nearby" (contain
    "near"), "corners" (contains "corner") and "desktop" (contains "top") as position words,
    which roughly doubles the result on this corpus.
    """
    pattern = r"\b(?:" + "|".join(re.escape(term) for term in terms) + r")\b"
    return series.str.lower().str.contains(pattern, regex=True)

`eda_viz` builds every figure. Each function returns a `Figure` and renders nothing, so the
axis labels, log scales and median markers are decided once rather than per cell.

In [ ]:
%%writefile eda_modules/eda_viz.py
"""Build the report's figures.

Why this module exists: a figure is only trustworthy if every one of them is labelled the
same way and uses the same scale for the same quantity. Building them here means the axis
labels, the log scale on skewed measures and the median line are decided once.

Each function returns a :class:`matplotlib.figure.Figure` and draws nothing itself, so the
caller decides when to render it and where it goes. No backend is forced: a notebook
wants its inline backend and a headless run wants Agg, and that is the caller's choice,
not this module's.
"""

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Marker colour used for the median line on every distribution figure.
MEDIAN_COLOUR = "darkred"


def plot_distributions(frame: pd.DataFrame, specs: list[tuple[str, str, bool]]) -> plt.Figure:
    """Draw a histogram and a box plot for each requested variable.

    Args:
        frame: the cleaned table.
        specs: one ``(column, axis label, log scale)`` tuple per variable.

    Returns:
        A figure with one row per variable and two columns.

    A histogram alone hides the outlier structure and a box plot alone hides multimodality,
    so both are drawn for the same variable in the same row. A log x-axis is applied where
    the caller asks for it, and the median is marked because on a skewed variable it is the
    summary being reported — a reader can see where it sits rather than take the number on
    trust.
    """
    figure, axes = plt.subplots(len(specs), 2, figsize=(11, 3 * len(specs)))
    axes = np.atleast_2d(axes)

    for row, (column, label, log_scale) in enumerate(specs):
        # Zero cannot be placed on a log axis, and a zero distance means the target is
        # flush with the edge rather than missing; such rows are excluded from the log view.
        values = frame[column].replace(0, np.nan) if log_scale else frame[column]

        sns.histplot(values.dropna(), kde=not log_scale, ax=axes[row, 0], color="steelblue")
        sns.boxplot(x=values.dropna(), ax=axes[row, 1], color="lightsteelblue")

        for axis in axes[row]:
            if log_scale:
                axis.set_xscale("log")
            axis.set_xlabel(label)
        axes[row, 0].set_ylabel("Grounding requests (count)")
        axes[row, 1].set_ylabel("")
        axes[row, 0].set_title(f"Distribution of {column} (n={int(values.notna().sum())})")
        axes[row, 1].set_title(f"Box plot of {column}")
        axes[row, 0].axvline(
            frame[column].median(), color=MEDIAN_COLOUR, linestyle="--", linewidth=1
        )

    figure.tight_layout()
    return figure


def plot_correlation(frame: pd.DataFrame, columns: list[str]) -> plt.Figure:
    """Draw a Spearman correlation heatmap for the requested numeric columns.

    Args:
        frame: the cleaned table.
        columns: numeric columns to correlate.

    Returns:
        A square heatmap figure with annotated coefficients.

    Rank correlation, because the size variables are skewed and the relations need not be
    linear; a Pearson heatmap on these columns would be dominated by a handful of large
    targets. Annotations are printed so a cell can be read without a colour legend, and
    the diverging palette is centred at zero so the sign is visible at a glance.
    """
    correlation = frame[columns].corr(method="spearman")

    figure, axis = plt.subplots(figsize=(8.5, 6.5))
    sns.heatmap(
        correlation,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        square=True,
        cbar_kws={"label": "Spearman rho"},
        ax=axis,
    )
    axis.set_title("Spearman rank correlation, numeric variables")
    axis.set_xlabel("")
    axis.set_ylabel("")
    figure.tight_layout()
    return figure


def plot_categoricals(frame: pd.DataFrame) -> plt.Figure:
    """Draw sorted count bars for application, industry group and target type.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with three panels, one per categorical variable.

    Bars are sorted by count so the imbalance is visible without reading the labels, and
    each bar carries its count or share. A share without its denominator would hide the
    fact that the smallest group holds 45 rows.
    """
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.4))

    application_counts = frame["application"].value_counts()
    sns.barplot(
        x=application_counts.to_numpy(), y=application_counts.index, ax=axes[0], color="steelblue"
    )
    axes[0].set_title(f"Requests per application (n={len(frame)})")
    axes[0].set_xlabel("Grounding requests (count)")
    axes[0].set_ylabel("Application")
    for position, value in enumerate(application_counts.to_numpy()):
        axes[0].text(value + 1, position, str(value), va="center", fontsize=8)

    group_counts = frame["group"].value_counts()
    sns.barplot(x=group_counts.index, y=group_counts.to_numpy(), ax=axes[1], color="seagreen")
    axes[1].set_title("Requests per industry group")
    axes[1].set_xlabel("Industry group")
    axes[1].set_ylabel("Grounding requests (count)")
    axes[1].tick_params(axis="x", rotation=30)
    for position, value in enumerate(group_counts.to_numpy()):
        axes[1].text(position, value + 4, str(value), ha="center", fontsize=8)

    type_counts = frame["ui_type"].value_counts()
    sns.barplot(x=type_counts.index, y=type_counts.to_numpy(), ax=axes[2], color="indianred")
    axes[2].set_title("Requests per target type")
    axes[2].set_xlabel("Annotated target type")
    axes[2].set_ylabel("Grounding requests (count)")
    for position, value in enumerate(type_counts.to_numpy()):
        share = value / len(frame) * 100
        axes[2].text(position, value + 4, f"{value} ({share:.1f}%)", ha="center", fontsize=8)

    figure.tight_layout()
    return figure


def plot_area_by_type(frame: pd.DataFrame) -> plt.Figure:
    """Compare target area and target width between icon and text targets.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with a log-scale box plot of relative area and a linear box plot of width.

    Two panels because the two measures answer different questions: relative area compares
    targets across screens of different sizes, while pixel width says how large the label
    physically is. Showing both prevents a resolution difference from being read as a
    target-type difference.
    """
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    sns.boxplot(
        data=frame,
        x="ui_type",
        y="target_area_pct",
        hue="ui_type",
        palette="Set2",
        legend=False,
        showfliers=False,
        ax=axes[0],
    )
    axes[0].set_yscale("log")
    axes[0].set_title("Target area by annotated target type (log scale)")
    axes[0].set_xlabel("Annotated target type")
    axes[0].set_ylabel("Target area (% of screen area, log scale)")

    sns.boxplot(
        data=frame,
        x="ui_type",
        y="bbox_w",
        hue="ui_type",
        palette="Set2",
        legend=False,
        showfliers=False,
        ax=axes[1],
    )
    axes[1].set_title("Target width by annotated target type")
    axes[1].set_xlabel("Annotated target type")
    axes[1].set_ylabel("Target width (pixels)")

    figure.tight_layout()
    return figure


def plot_area_by_application(frame: pd.DataFrame) -> plt.Figure:
    """Draw target area per application, split by target type.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with applications ordered by median target area.

    Ordering by median rather than alphabetically puts the comparison the report makes
    first: the smallest and largest typical targets are at the two ends. Splitting by type
    inside each application shows whether the icon/text gap is consistent or produced by
    one application's composition.
    """
    order = frame.groupby("application")["target_area_pct"].median().sort_values().index

    figure, axis = plt.subplots(figsize=(11, 5.6))
    sns.boxplot(
        data=frame,
        x="application",
        y="target_area_pct",
        order=order,
        hue="ui_type",
        palette="Set2",
        showfliers=False,
        ax=axis,
    )
    axis.set_yscale("log")
    axis.set_title("Target area by application and target type (log scale)")
    axis.set_xlabel("Application (ordered by median target area)")
    axis.set_ylabel("Target area (% of screen area, log scale)")
    axis.tick_params(axis="x", rotation=35)
    axis.legend(title="Target type", bbox_to_anchor=(1.01, 1), loc="upper left")

    figure.tight_layout()
    return figure


def plot_resolution_relationship(frame: pd.DataFrame, medians: pd.DataFrame) -> plt.Figure:
    """Draw the resolution-versus-target-size relation, pooled and per resolution.

    Args:
        frame: the cleaned table.
        medians: one row per resolution with ``screen_label`` and ``target_area_pct``.

    Returns:
        A figure with a pooled scatter and a bar chart of per-resolution medians.

    The pooled panel invites a resolution story and the bar panel tests it: if the pooled
    trend came from a few points, the per-resolution bars will show that the relation is
    not monotone and the claim has to be softened.
    """
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    sns.scatterplot(
        data=frame,
        x="screen_megapixels",
        y="target_area_pct",
        hue="ui_type",
        alpha=0.45,
        s=18,
        palette="Set2",
        ax=axes[0],
    )
    axes[0].set_yscale("log")
    axes[0].set_title("Target area vs screen resolution (pooled)")
    axes[0].set_xlabel("Screen resolution (megapixels)")
    axes[0].set_ylabel("Target area (% of screen area, log scale)")

    sns.barplot(
        data=medians, x="screen_label", y="target_area_pct", ax=axes[1], color="steelblue"
    )
    axes[1].set_title("Median target area by native resolution")
    axes[1].set_xlabel("Native screen resolution (pixels)")
    axes[1].set_ylabel("Median target area (% of screen area)")
    axes[1].tick_params(axis="x", rotation=35)

    figure.tight_layout()
    return figure


def plot_type_share(frame: pd.DataFrame) -> plt.Figure:
    """Draw the icon/text mix within each application as a stacked share bar.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with one horizontal stacked bar per application.

    Row percentages, not counts, because the question is composition: a bar of 82
    PowerPoint requests and a bar of 31 Illustrator requests are only comparable once both
    are scaled to their own total.
    """
    shares = pd.crosstab(frame["application"], frame["ui_type"], normalize="index") * 100
    shares = shares.sort_values("icon")

    figure, axis = plt.subplots(figsize=(11, 5.4))
    shares.plot(
        kind="barh", stacked=True, ax=axis, color=["indianred", "steelblue"], width=0.8
    )
    axis.set_title("Share of icon vs text targets within each application (row percentages)")
    axis.set_xlabel("Share of requests within the application (%)")
    axis.set_ylabel("Application")
    axis.legend(title="Target type", loc="lower right")
    axis.axvline(50, color="black", linestyle=":", linewidth=1)

    figure.tight_layout()
    return figure


def plot_instruction_length(frame: pd.DataFrame) -> plt.Figure:
    """Draw instruction length distributions in words and characters.

    Args:
        frame: the cleaned table.

    Returns:
        A figure with both histograms and the median marked.

    Words and characters are shown together because their ratio is informative: many words
    of few characters each is a different instruction style from few long words, and only
    the pair distinguishes them.
    """
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    sns.histplot(
        frame["instruction_words"], bins=range(1, 18), ax=axes[0], color="steelblue"
    )
    axes[0].set_title("Instruction length in words")
    axes[0].set_xlabel("Words per instruction")
    axes[0].set_ylabel("Grounding requests (count)")
    axes[0].axvline(
        frame["instruction_words"].median(), color=MEDIAN_COLOUR, linestyle="--", linewidth=1
    )

    sns.histplot(frame["instruction_chars"], bins=30, ax=axes[1], color="seagreen")
    axes[1].set_title("Instruction length in characters")
    axes[1].set_xlabel("Characters per instruction")
    axes[1].set_ylabel("Grounding requests (count)")
    axes[1].axvline(
        frame["instruction_chars"].median(), color=MEDIAN_COLOUR, linestyle="--", linewidth=1
    )

    figure.tight_layout()
    return figure


def plot_top_terms(terms: pd.DataFrame) -> plt.Figure:
    """Draw the most frequent instruction terms as a sorted bar chart.

    Args:
        terms: output of :func:`isdm.eda.analysis.top_terms`.

    Returns:
        A horizontal bar figure ordered from least to most frequent.
    """
    figure, axis = plt.subplots(figsize=(8, 5))
    sns.barplot(
        data=terms.sort_values("count"), x="count", y="term", color="steelblue", ax=axis
    )
    axis.set_title("Most frequent instruction terms after stop-word removal")
    axis.set_xlabel("Occurrences (count)")
    axis.set_ylabel("Term")
    figure.tight_layout()
    return figure


def plot_target_crops(
    frame: pd.DataFrame, image_root: Path, applications: list[str], n: int = 2, seed: int = 42
) -> plt.Figure:
    """Crop around the annotated target in sample screenshots and outline the box.

    Args:
        frame: the cleaned table.
        image_root: directory the ``img_filename`` values are relative to.
        applications: applications to sample from, one row of panels each.
        n: screenshots per application.
        seed: sampling seed so the same examples are drawn on every run.

    Returns:
        A figure with one row per application and ``n`` crops per row.

    The crop is the point. A full 5120x1440 screenshot renders a 16-pixel icon as a
    handful of pixels, so the annotated box would be invisible and the check meaningless.
    Zooming to a window around the box makes it possible to see whether the box is on the
    requested element — the one check no summary statistic can perform.
    """
    from PIL import Image

    figure, axes = plt.subplots(
        len(applications), n, figsize=(7 * n, 3.6 * len(applications))
    )
    axes = np.atleast_2d(axes)

    for row, application in enumerate(applications):
        subset = frame[frame["application"] == application].sample(
            min(n, len(frame[frame["application"] == application])), random_state=seed
        )
        for column in range(n):
            axis = axes[row, column]
            if column >= len(subset):
                axis.axis("off")
                continue

            record = subset.iloc[column]
            with Image.open(image_root / record["img_filename"]) as image:
                axis.imshow(image.convert("RGB"))

            axis.add_patch(
                plt.Rectangle(
                    (record["bbox_x1"], record["bbox_y1"]),
                    record["bbox_w"],
                    record["bbox_h"],
                    fill=False,
                    edgecolor="red",
                    linewidth=1.8,
                )
            )

            # Keep a fifth of the screen as context on each side of the target.
            pad_x = record["img_size_w"] * 0.22
            pad_y = record["img_size_h"] * 0.22
            axis.set_xlim(
                max(0, record["bbox_x1"] - pad_x),
                min(record["img_size_w"], record["bbox_x2"] + pad_x),
            )
            axis.set_ylim(
                min(record["img_size_h"], record["bbox_y2"] + pad_y),
                max(0, record["bbox_y1"] - pad_y),
            )
            axis.set_title(
                f"{record['application']} | {record['ui_type']} | {record['instruction'][:32]}",
                fontsize=9,
            )
            axis.set_xlabel("Screen x (pixels)")
            axis.set_ylabel("Screen y (pixels)")

    figure.tight_layout()
    return figure


def plot_pairplot(
    frame: pd.DataFrame,
    columns: list[str],
    hue: str,
    sample_size: int = 500,
    seed: int = 42,
) -> sns.PairGrid:
    """Draw a pairwise grid of several variables coloured by one categorical.

    Args:
        frame: the cleaned table.
        columns: numeric or categorical columns to include, four to six is readable.
        hue: categorical column to colour by.
        sample_size: rows to draw; scatter plots of close to a thousand points become a
            solid mass.
        seed: sampling seed.

    Returns:
        The :class:`seaborn.PairGrid`, whose ``figure`` holds the axes.

    Sampling is a plotting decision only. Every statistic in the report is computed on all
    rows, and the sample size is stated in the figure title so a reader cannot mistake the
    grid for the analysis.
    """
    sample = frame.sample(min(sample_size, len(frame)), random_state=seed)
    grid = sns.pairplot(
        sample[columns],
        hue=hue,
        palette="Set2",
        corner=True,
        plot_kws={"s": 12, "alpha": 0.5},
        diag_kind="kde",
    )
    grid.figure.suptitle(
        f"Pairwise relationships, {len(sample)}-row random sample (seed={seed})", y=1.01
    )
    return grid

The four modules are on disk beside the notebook. Importing them now proves the `%%writefile`
path worked and that the notebook is genuinely self-contained rather than depending on
something installed in this environment.

In [ ]:
import eda_analysis as analysis
import eda_features as features
import eda_quality as quality
import eda_viz as viz

from importlib.metadata import version

# Every third-party library this notebook uses, including the two beyond the assignment's
# setup cell: Pillow opens the screenshots, tabulate renders the cleaning log as Markdown.
THIRD_PARTY = ("pandas", "numpy", "matplotlib", "seaborn", "scipy", "pillow", "tabulate")

print("python:", sys.version.split()[0])
for package in THIRD_PARTY:
    print(f"{package:<12}: {version(package)}")
print("modules from:", Path(quality.__file__).parent)

The modules imported from the local `eda_modules` directory, and the versions printed above are
the ones this analysis ran on. **That cell is also this submission's library listing** as the
assignment requires: seven third-party packages, two of which go beyond the setup cell — Pillow
(opens the screenshots) and tabulate (renders the cleaning log as a Markdown table). Everything
else imported here is Python standard library.

The versions matter for reproducibility: seaborn 0.13 is required for the `legend=` argument used
in the box plots, and pandas 2.2 or later for the named aggregation in `application_medians`.

Loading next. `id` is read as a string on purpose — it is an identifier, so reading it as a
number would invite a meaningless average — and the four pixel columns are forced to integer
so a numeric-looking column cannot arrive as text.

In [ ]:
df = pd.read_csv(
    DATA_DIR / "grounding_requests.csv",
    dtype={"id": str, "application": str, "group": str, "platform": str,
           "ui_type": str, "img_filename": str},
)

INTEGER_COLUMNS = ["img_size_w", "img_size_h", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]
df[INTEGER_COLUMNS] = df[INTEGER_COLUMNS].astype("int64")

print("shape (rows, columns):", df.shape)
print("distinct requests    :", df["id"].nunique())
print("distinct filenames   :", df["img_filename"].nunique())
print("applications         :", df["application"].nunique())
print("industry groups      :", df["group"].nunique())

**1,020 rows, 1,020 distinct request ids, 1,020 distinct filenames, 17 applications, 6 industry
groups.** One row per request and one request per filename, so at this level the corpus is
tidy. Whether those 1,020 filenames represent 1,020 distinct *screens* is a different question,
answered in section 3 — and the answer is no.

A head alone is not enough: the files were concatenated in alphabetical order by application,
so the top rows are all AutoCAD. A random sample is the honest first look.

In [ ]:
display(df.head(3))

print("random sample:")
display(df.sample(5, random_state=RANDOM_STATE))

The head is all AutoCAD, exactly as the concatenation order predicts, while the random sample
mixes applications and resolutions. In the sample, `bbox` coordinates run to the thousands
against `img_size` of 3840x1080 (or 5120x1440), so the coordinate space is the screenshot's
own pixel grid rather than a normalised 0–1000 scale. Section 3 confirms that against the
files themselves rather than assuming it.

`info()` is the type and null-count check the assignment asks for, run before any summary.

In [ ]:
df.info()

Every column is non-null, the identifier stayed a string, and the pixel columns are integers.
There is no date column in this release, so section 7 declares the time-series checks not
applicable rather than improvising them from the timestamps buried in the filenames.

`describe(include="all")` next, to see the range of every column at once. Eight of the
fourteen columns are labels, so their means are meaningless; the table is read for min, max
and level counts.

In [ ]:
display(df.describe(include="all").T)

Two things stand out. `bbox_y1` has a minimum of **-1**, which is not a valid screen
coordinate, and `instruction` has a minimum length of 2 characters, so some instructions are
single short words. The first is a defect to repair in section 4; the second is a property of
the task that limits how far the text can disambiguate a target.

Before any analysis, the merged table is checked against the 16 raw annotation files it came
from. The merge is the one step that can lose or duplicate rows silently, and every number
downstream would inherit that error.

In [ ]:
verification = quality.requests_vs_annotations(df, DATA_DIR / "annotations")
display(verification)

print("requests in raw files :", int(verification["records"].sum()))
print("requests in merged CSV:", len(df))
print("total difference      :", int(verification["difference"].sum()))

The raw files hold exactly 1,020 records and the merged table holds 1,020 rows, with a
difference of zero in every file, so the merge is lossless and no request was dropped or
counted twice.

The totals also reconcile with the paper, which is how the MATLAB problem surfaced. The
paper's table lists 15 named Windows applications totalling 846 requests, plus MATLAB at 93,
plus the 81 `windows_common` requests: 1,020. The release files MATLAB under a macOS name
instead (`matlab_macos.json`, `images/matlab_mac/`, `"platform": "macos"`), so reading the
filenames alone gives 927 and silently drops a whole application. Section 3 establishes which
label is right, and the correction is logged in section 4.

## 3. Data quality assessment

Each problem is measured before anything is changed, so the cleaning log can state a real
count instead of an impression. The checks are: missingness, duplicates, categorical levels,
bounding-box validity, outliers, resolution support, and the screenshot files themselves.

Missing values first, counted and as a percentage — a single missing cell in 1,020 rows is
0.1%, which is easy to misread as "no missing data" if only counts are shown.

In [ ]:
missing = quality.missing_table(df)
display(missing.T)

print("missing cells            :", int(df.isna().sum().sum()))
print("rows with any missing    :", int(df.isna().any(axis=1).sum()))

There are **no missing values anywhere**: 0 cells, 0 rows. The usual missingness-pattern
analysis — is income missing mostly in one region? — therefore has nothing to work on. That
is a real property of an expert-annotated benchmark with a fixed schema, not a skipped check,
and it means every row is available for analysis. It also means the data cannot tell us how
often a real capture pipeline fails to emit a usable annotation, which is recorded in the
limitations.

Duplication is checked at three levels, because they mean different things: a duplicated row
is a record entered twice, a duplicated id is a request named twice, and a repeated
instruction string is neither.

In [ ]:
duplicates = quality.duplicate_report(df)
display(duplicates.to_frame())

print()
reused = df[df["instruction"].duplicated(keep=False)].sort_values("instruction")
print("distinct instruction strings:", df["instruction"].nunique(), "of", len(df), "requests")
display(reused[["id", "application", "ui_type", "instruction"]].head(8))

No duplicate rows and no duplicate ids, so nothing is dropped on that account. But **60 rows
share an instruction string with another row**, and those 60 rows involve 25 distinct strings,
so only 985 of the 1,020 instructions are unique. Examples: *"Copy"* in both Quartus and
AutoCAD, *"Draw a circle"* in Inventor and SolidWorks, sometimes naming an icon in one
application and a text label in another.

This is not a cleaning defect and the rows must be kept. It is a substantive property: the
same instruction can name different targets in different applications, so **instruction text
cannot be used as a key to a target**. That matters for any later error analysis, and it is
carried into the limitations rather than repaired.

Categorical levels are audited next for typographical and whitespace variants, which split a
category in two and halve every later share without any error being raised.

In [ ]:
levels = quality.level_audit(df, ("application", "group", "platform", "ui_type"))
display(levels)

print()
print("instructions with leading/trailing whitespace:",
      int((df["instruction"] != df["instruction"].str.strip()).sum()))
print("instructions with double spaces              :",
      int(df["instruction"].str.contains("  ", regex=False).sum()))

The label columns are clean in themselves: no stray whitespace, no casing variants, and the
applications use the short form (`powerpoint`, not `PowerPoint`) consistently across all 1,020
rows. `platform` is the exception and the reason this section is not the end of the story: it
arrives with **two** levels, 927 `windows` and 93 `macos`, and every one of the `macos` rows
belongs to MATLAB. A benchmark described as a Windows subset that reports a second platform
for one application is either a mixed-platform corpus or a mislabel, and the next check
settles which.

The instruction text is the exception, and the output above shows it: **5 requests carry a
leading or trailing space and 4 contain a doubled space**. That is small but real, it inflates
the character count that section 5 reports, and it is repaired and logged in section 4.

Bounding-box validity is checked against what is logically possible: corners ordered,
non-zero area, and the box inside the declared screen. These are the checks that catch an
annotation entered in the wrong coordinate space.

In [ ]:
counts, offending = quality.bbox_violations(df)
display(counts)

print("offending rows:", len(offending))
display(offending[["id", "application", "img_size_w", "img_size_h",
                   "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]])

Exactly **one row is invalid**: the Inventor request `inventor_windows_60`, whose box starts at
`y1 = -1` on a 3840x1080 screen. The box is otherwise an ordinary 16x20 icon flush against the
top edge, so the annotation is off by one pixel at the boundary rather than written in a
different coordinate system. No box is inverted, none has zero area, and none exceeds its
screen — which is what establishes that the corner convention is consistent across the whole
corpus.

Outliers come next. The IQR rule flags values; it does not delete them. The assignment is
explicit that unusual values are inspected first, and this dataset makes that advice
concrete in section 6.

In [ ]:
# The outlier rule needs the derived size columns, so build them once here rather than
# recomputing per test. `clean` in section 4 is this frame plus the documented repairs.
geometry = features.add_derived_features(df)

outliers = quality.outlier_summary(geometry, ("bbox_w", "bbox_h", "target_area_pct"))
display(outliers)

flagged, lower, upper = quality.iqr_flags(geometry["target_area_pct"])
print("upper fence (%):", round(upper, 4))
largest = geometry.loc[flagged].nlargest(8, "target_area_pct")
display(largest[["id", "application", "ui_type", "bbox_w", "bbox_h", "target_area_pct"]])

The IQR rule flags **41 rows on target area (4.0% of the corpus)**, 55 on width and 149 on
height. The largest is a 2060x119 region covering 4.73% of its screen: a band across the interface
rather than a button. The 430x284 region is second at 1.66%.

None of these is an error. An instruction can legitimately name a large region, and these are
the *easiest* cases in the corpus: deleting them would inflate the apparent difficulty of the
benchmark and bias every summary. They are kept, and the analysis uses medians and rank-based
tests because the size distribution is so skewed.

Resolution is not an outlier question but a support question: a resolution carried by a
handful of rows cannot support a per-resolution claim.

In [ ]:
support = quality.resolution_support(df)
display(support)
print("distinct resolutions:", len(support))

Seven resolutions, unevenly used. **2560x1440 dominates with 430 requests (42.2%)**, followed
by the ultrawide 3840x1080 at 181 (17.8%) and 3840x2160 at 148 (14.5%). The most familiar
desktop size, 1920x1080, carries only **19 requests (1.9%)** — so any statement of the form
"on a 1080p screen" would rest on under 2% of the data. MATLAB's 93 requests are all
2560x1440, which is part of why that resolution grew.

The annotations declare `img_size`, but a declared size is not evidence. Opening every file
establishes three facts at once: it exists, it decodes, and its pixel dimensions match the
coordinate space the boxes were written in.

In [ ]:
audit, joined = quality.audit_images(df, DATA_DIR / "images")
print("status:", audit["status"].value_counts().to_dict())
print("colour modes:", audit["mode"].value_counts().to_dict())
print("distinct screenshots:", audit["img_filename"].nunique())

mismatch = quality.size_mismatch(joined)
print("files whose real size differs from the annotation:", len(mismatch))
display(mismatch[["img_filename", "img_size_w", "img_size_h", "file_w", "file_h"]])

total_bytes = sum(p.stat().st_size for p in (DATA_DIR / "images").rglob("*.png")) / 1e6
print("total image volume:", round(total_bytes, 1), "MB")

All 1,020 files exist, all open, all are RGB with no alpha channel, and **every file's real
pixel size matches its annotation exactly** (0 mismatches). That is the most important result
in this section: it is what makes a relative measure like "share of screen area" mean
anything, because the boxes and the image share one pixel grid.

Duplication is now checked twice, and the two checks disagree.
A perceptual hash is cheap and is the usual tool; a SHA-256 digest is exact and cannot be
fooled.

In [ ]:
files = sorted(audit["img_filename"].unique())
digests = quality.byte_duplicates(files, DATA_DIR / "images")
print("files sharing their bytes with another file:", len(digests))
print("distinct duplicate groups:", digests["sha256"].nunique())
display(digests.head(6))

**28 files are byte-identical to another file**, forming 14 duplicate pairs, so the 1,020
requests describe **992 distinct screens**. So filenames being unique did *not* mean the
screens were unique.

Each pair is within one application, the two captures are 9–36 seconds apart by filename,
and the two requests carry **different instructions**: `vivado_windows_4` asks for "add report
in vivado" and `vivado_windows_9` for "search report in vivado", on the same image. The screen is asked about twice, which is legitimate benchmark design, but it means 28
requests are not independent of each other, and it is the kind of dependence the project's own
analysis-unit decision warns about.

The perceptual hash is checked next, and fails. Reporting the failure is the point: a hash
that looks like it worked would have supported a false "no near-duplicates" claim.

In [ ]:
hashes = np.array([quality.average_hash(DATA_DIR / "images" / name) for name in files])
distance_summary = quality.hash_distance_summary(files, hashes)
display(distance_summary)

identical = int(distance_summary.loc[distance_summary["max_distance"] == 0, "pairs"].iloc[0])
print(f"bit-identical hash pairs: {identical} of {int(distance_summary['total_pairs'].iloc[0])}")
print("cross-application pairs within 2 bits:",
      int(distance_summary.loc[distance_summary["max_distance"] == 2, "cross_application"].iloc[0]))

The average hash reports **2,554 bit-identical pairs** among 519,690 — far more than the 14
real duplicate pairs. Of the 3,990 pairs within 2 bits, 3,920 are inside one application and
**70 cross an application boundary**, 30 of them bit-identical.

The cross-application column is the diagnostic, and it is no longer zero. An 8x8 greyscale
thumbnail mostly encodes window chrome and panel layout, which captures of one application
share — but two applications with similar chrome can collide as well, which is exactly what
30 bit-identical cross-application pairs show. The hash is measuring
*layout*, not content, and it cannot distinguish two different documents open in Blender. It
is therefore discarded, and the byte comparison above is the duplicate check the analysis
relies on. Recording the failure is worth more than the hash itself: a perceptual hash applied
to GUI screenshots will report near-duplicates everywhere and a reader has no way to tell
whether that means anything.

The quality assessment closes with one off-by-one coordinate, no missingness, no duplicate
rows or ids, no corrupt files, 14 genuinely duplicated screens, one mislabelled platform that
the captures contradict, and a perceptual hash that does not discriminate on this data.

## 4. Data cleaning

Cleaning runs on a copy so the raw frame stays intact and every later number is traceable to
an unmodified source. Each change appends a row to a log, and the log is rendered as a table
at the end of the section. The steps and their order are in `eda_features.clean_requests`.

In [ ]:
clean, cleaning_log = features.clean_requests(df)

print("rows before cleaning:", len(df))
print("rows after  cleaning:", len(clean))
print("columns            :", df.shape[1], "->", clean.shape[1])
print()
display(cleaning_log)

**1,020 rows before and after, 29 columns instead of 14.** Three repairs, none of them
destructive of an observation:

- **93 MATLAB records were labelled `macos`** and are Windows captures. The evidence is recorded
  rather than asserted: sampled screenshots — for example
  `matlab_mac/screenshot_2024-11-30_20-47-44.png` — show a Windows 10 taskbar, right-hand window
  controls and a `D:\annotation` working directory in the folder pane, and the paper's own table
  lists MATLAB as a Windows application. The label was wrong in the record, the filename and the
  image directory at once.
- **8 instructions had stray whitespace** (5 with a leading or trailing space, 4 with a doubled
  space, one row with both). Stripping it changes a character count by one or two, which would
  otherwise be an unexplained wobble in section 5's length statistics.
- **1 coordinate fell outside the screen** and was clamped to 0. Dropping the row would have cost
one Inventor icon request, and Inventor contributes 70 rows in total, so deleting would have
removed 1.4% of one application's evidence to fix one pixel.

No repair discards an observation, so the row count is unchanged. The final log line records
the checks that found nothing — deliberate, because a log with only entries for repairs is
indistinguishable from a cleaning step that was never performed.

The same log as a Markdown table, for the written record:

In [ ]:
print(cleaning_log.to_markdown(index=False))

The log is complete: three changes, each with its reason and row count, one explicit no-action
entry, and the audit that produced them is in section 3. Per the assignment, cleaning is iterative —
nothing in sections 5 to 7 sent the analysis back here.

The derived features are constructed once, immediately after cleaning, so that every later
cell shares one definition of "target size". `add_derived_features` builds them and explains
each one; the summary below is the check that they came out sane.

In [ ]:
DERIVED = list(features.DERIVED_COLUMNS)
display(clean[DERIVED].describe().T.round(4))

print("target area range (%): ", round(clean["target_area_pct"].min(), 4), "to",
      round(clean["target_area_pct"].max(), 4))
print("median target area (%):", round(clean["target_area_pct"].median(), 4))

`target_area_pct` spans **0.002% to 4.73%** with a median of **0.0382%** — a range of more
than three orders of magnitude in one variable. `edge_distance_pct` runs from 0 (target flush
against a screen edge) to 49.1, and `instruction_words` from 1 to 16.

Two things are deliberately *not* built here. There is no tokenised vocabulary and no element
taxonomy beyond `icon`/`text`; both need decisions (stemming? which stop-words? which element
classes?) that belong to the modelling phase. Inventing them during cleaning would bake
analysis choices into a step the report presents as neutral preparation.

## 5. Univariate analysis

One variable at a time, with the summary chosen to fit the variable's type. Numeric variables
get centre, spread and shape; because most of them are strongly right-skewed, skewness is
reported beside the mean so that the mean's fitness can be judged rather than assumed.

In [ ]:
NUMERIC = ["screen_megapixels", "bbox_w", "bbox_h", "target_area_pct",
           "bbox_w_pct", "bbox_h_pct", "edge_distance_pct",
           "instruction_chars", "instruction_words"]
display(analysis.numeric_summary(clean, NUMERIC))

Every size variable is strongly right-skewed: `target_area_pct` has a skewness of **20.3**, its
mean (0.0653%) is about 1.7 times its median (0.0382%), and `bbox_w` skews at 6.8 with a mean
of 91.6 px against a median of 49 px.

This is the case the assignment warns about. Reporting the mean of `target_area_pct`
would describe a target size that barely exists. **The median and IQR are used from here on**,
and group comparisons use rank-based tests.

`screen_megapixels` is the exception at skew 1.20, because it is not a continuous measurement
at all but a small set of fixed resolutions — its distribution should look like spikes, and
the next figure tests that.

In [ ]:
SPECS = [
    ("screen_megapixels", "Screen resolution (megapixels)", False),
    ("target_area_pct", "Target area (% of screen area, log scale)", True),
    ("bbox_w_pct", "Target width (% of screen width, log scale)", True),
    ("edge_distance_pct", "Target distance to nearest screen edge (% of short side)", False),
]
viz.plot_distributions(clean, SPECS)
plt.show()

print("targets flush against a screen edge (distance 0):",
      int((clean["edge_distance_pct"] == 0).sum()))

The resolution histogram is spiky, exactly as a set of seven fixed configurations should be,
with peaks at 4.1 MP (2560x1440) and the 3840-wide variants. This subset is not a sample of
panel types; it reflects a handful of real capture machines.

`target_area_pct` on a log scale is close to symmetric: **target
area is roughly log-normally distributed**, spanning about 0.002% to 4.7% of the screen. The
median of 0.0382% on a 1920x1080 screen is about 790 square pixels — smaller than a 29x29
button. The box plot's long right tail is produced by the large regions already identified as
legitimate rather than erroneous.

`edge_distance_pct` is visibly two-part: a cluster near 0–2% (targets flush against a window
border or screen edge — 17 rows sit exactly at 0) and the rest spread across the interior.
Targets at the screen boundary are a different grounding problem from small interior targets,
and `bbox_w_pct` confirms that most targets sit between 1% and 6% of screen width.

Categorical variables next: counts, sorted, with group sizes shown, because a share without
its denominator hides the smallest group.

In [ ]:
viz.plot_categoricals(clean)
plt.show()

print(clean["ui_type"].value_counts(normalize=True).mul(100).round(1).to_string())

Per application the corpus is uneven but workable: PowerPoint contributes 82 requests and
Illustrator 31, a 2.6x spread. Per group it is severe. Creative holds **297 requests (29.1%)** across 6 applications while
Dev holds **45 (4.4%)** in one, a 6.6x spread. So
"professional software has property X" is, in this corpus, mostly a statement about Creative
applications, and a per-group average is an average over unequal samples.

Per target type the split is **622 text targets (61.0%)** against **398 icons (39.0%)**, close
enough that the comparison in section 6 is worth making, and much better balanced than a
34-cell application-by-type breakdown would be, where 12 of the 34 cells hold fewer than 20 rows.

## 6. Bivariate and multivariate analysis

Relationships are read from the plot that fits the pair of variable types, with a number
attached. Rank correlation is used throughout because the size variables are heavily skewed
and the relationship need not be linear.

A correlation matrix first, as an overview rather than an answer. A heatmap alone is one of the
assignment's named pitfalls, so nothing below rests on it.

In [ ]:
CORRELATED = ["screen_megapixels", "bbox_w", "bbox_h", "target_area_pct",
              "bbox_w_pct", "bbox_h_pct", "edge_distance_pct", "instruction_chars"]
viz.plot_correlation(clean, CORRELATED)
plt.show()

The strongest cells are arithmetic: `bbox_w`, `bbox_h` and `target_area_pct` move together
because area is their product, so those cells carry no information. The informative ones are
that `screen_megapixels` is weakly negative against the size measures and that
`instruction_chars` is nearly independent of every geometric variable.

`bbox_w` correlates 0.96 with `bbox_w_pct` under Spearman, so at these resolutions the two rank
targets almost identically and neither can be used to argue the other is misleading. Relative
area is still the measure the analysis uses, for a reason the correlation cannot show: it is
the one that stays comparable if a higher-resolution capture is added later.

Q1 next: target size by target type, with a rank test, the group medians, and an effect size.

In [ ]:
viz.plot_area_by_type(clean)
plt.show()

icon_area = clean.loc[clean["ui_type"] == "icon", "target_area_pct"]
text_area = clean.loc[clean["ui_type"] == "text", "target_area_pct"]
comparison = analysis.rank_comparison(icon_area, text_area, "icon", "text")
for key, value in comparison.items():
    print(f"{key:>16}: {value}")

Icon and text targets are **not** comparable in size. The median icon covers **0.0144%** of its
screen against **0.0684%** for the median text target — a **4.7x** difference (Mann-Whitney
U = 29,449, p = 7.2e-94, rank-biserial **+0.762**, which on this convention means icons sit
almost entirely below text targets in the ranking).

Text targets are wider because they are labels: their boxes are wide and thin (median aspect
3.62) while icon boxes are closer to square (median aspect 1.05), and the right-hand panel
shows the width gap directly.

This is the central descriptive result for Q1. Roughly 39% of the corpus is icon targets, and
icons are the ones occupying thousandths of a percent of a high-resolution screen. A single
aggregate "target size" for this benchmark would average two populations that differ 4.7x in
median size, so size must be reported per target type.

Q2 next: does the corpus's balance hold up at the application level, and how much of the
variation lives between applications?

In [ ]:
app_medians = analysis.application_medians(clean, "target_area_pct")
display(app_medians)

smallest, largest_app = app_medians.index[0], app_medians.index[-1]
print(f"smallest median: {smallest} = {app_medians['median'].iloc[0]:.4f}%")
print(f"largest  median: {largest_app} = {app_medians['median'].iloc[-1]:.4f}%")
print(f"ratio           = {app_medians['median'].iloc[-1] / app_medians['median'].iloc[0]:.1f}x")

viz.plot_area_by_application(clean)
plt.show()

Median target size differs by **11.8x** across applications: Origin's median is **0.0067%** of
its screen and MATLAB's is **0.0783%**. Two mechanisms are visible and they do not cancel.

The first is interface style: Origin and Stata are plotting tools whose annotators point at
small toolbar icons, while SolidWorks annotates feature-manager rows and MATLAB — also a
plotting tool, at 79.6% text targets — annotates menu and panel labels that are far larger. The
second is target-type mix, and the plot shows it clearly — where an application has both types,
text targets sit above icons almost everywhere. So the icon/text gap from Q1 is not produced by
one atypical application.

Q3 next: resolution against relative target size. The pooled correlation is easy; whether it
can be separated from application is the real question.

In [ ]:
res_medians = (clean.groupby(["img_size_w", "img_size_h", "screen_megapixels"])
               ["target_area_pct"].median().reset_index())
res_medians["screen_label"] = (res_medians["img_size_w"].astype(str) + "x"
                               + res_medians["img_size_h"].astype(str))

viz.plot_resolution_relationship(clean, res_medians)
plt.show()

display(res_medians[["screen_label", "screen_megapixels", "target_area_pct"]].round(4))
print(analysis.spearman_summary(clean, "screen_megapixels", "target_area_pct"))

Pooled, higher resolution is associated with a **smaller** relative target, but weakly:
Spearman rho = **-0.088** (p = 0.0048). The per-resolution medians are not monotone: 3840x2160
sits at 0.0153% while 3840x1080 sits at 0.0577%, and 1920x1080 at 0.0278% on 19 rows.

A correlation of -0.09 with p = 0.005 at n = 1,020 is the kind of result that is statistically
detectable and practically negligible. Before concluding anything, the association is checked
inside each application, because screens are nested in applications and application choice may
determine both resolution and target size.

In [ ]:
subgroups = analysis.subgroup_correlations(
    clean, "application", "screen_megapixels", "target_area_pct")
display(subgroups)

varied = int((subgroups["distinct_x"] > 1).sum())
print("applications with more than one native resolution:", varied, "of", len(subgroups))
print("applications that can support a within-group correlation:",
      int(subgroups["rho"].notna().sum()))

The subgroup check produces a structural result rather than a weaker one.
Fifteen of the seventeen applications were captured at exactly **one** native resolution;
Quartus and Windows Common have two, and MATLAB's 93 requests are all 2560x1440. Resolution is therefore (nearly) constant *within* every
application, which means **resolution and application are confounded to the point of being
inseparable in this subset**.

That changes the answer to Q3. It is not that the resolution effect is a composition effect
that shrinks inside groups; it is that a within-application comparison does not exist here at
all. The pooled rho of -0.088 is a between-application comparison in disguise, and no amount
of analysis of this release can say whether a larger screen makes targets relatively smaller.
That would need one application captured at several resolutions — a controlled condition the
project's own test matrix could produce, and this benchmark does not contain.

Q2's remaining piece is the target-type mix by application, which is what makes an
application-level average ambiguous.

In [ ]:
shares, chi_square = analysis.crosstab_share(clean, "application", "ui_type")
display(shares.sort_values("icon", ascending=False))

print()
for key, value in chi_square.items():
    print(f"{key:>14}: {value}")

viz.plot_type_share(clean)
plt.show()

The target-type mix is **not** constant across applications (chi-square = **192.9**, dof = 16,
p = 2.2e-32, and every expected cell above 12, so the test's assumption holds). Stata is **83.7%**
icon targets and EViews 14.0%; Quartus is 71.1% icon, Illustrator 61.3%, while Inventor,
SolidWorks and Unreal Engine sit near 16–18% and MATLAB at 20.4%.

This is the mechanism behind the application-level spread in Q2. Comparing Stata's median
target with Illustrator's compares a corpus of toolbar icons with a corpus of text labels, so
any application-level average has to be read together with its type mix, and Q1 is reported
per type for that reason.

Q4 next: instruction text against target type.

In [ ]:
viz.plot_instruction_length(clean)
plt.show()

display(analysis.instruction_summary(clean))
print(analysis.spearman_summary(clean, "instruction_words", "target_area_pct"))

icon_words = clean.loc[clean["ui_type"] == "icon", "instruction_words"]
text_words = clean.loc[clean["ui_type"] == "text", "instruction_words"]
words_by_type = analysis.rank_comparison(icon_words, text_words, "icon", "text")
print()
print("instruction words, icon vs text:", words_by_type["left_median"], "vs",
      words_by_type["right_median"], "| p =", round(words_by_type["p_value"], 4))

Instructions are short: a median of **4 words and 20 characters**, with **78 requests (7.6%)**
using a single word. The word histogram peaks at 3–5 and falls away sharply.

Length does not distinguish the two target types in any meaningful way (median 4 words for both,
p = 0.45, rank-biserial -0.03), and it does not predict target size (rho = -0.023, p = 0.46).
Both are useful negative results: a hard target is **not** signalled
by a longer instruction, so instruction length is not a difficulty feature, and any plan that
assumed otherwise would have been building on nothing.

A multivariate view closes the section, limited to the readable size of a pair grid.

In [ ]:
PAIR_COLUMNS = ["target_area_pct", "bbox_w_pct", "edge_distance_pct",
                "instruction_words", "ui_type"]
viz.plot_pairplot(clean, PAIR_COLUMNS, hue="ui_type", sample_size=500, seed=RANDOM_STATE)
plt.show()

The pair grid reproduces the two findings that survive every view: text targets sit higher on
`target_area_pct` and `bbox_w_pct` at almost any instruction length, and instruction length
itself separates nothing. The icon cluster is a tighter, smaller band throughout.

The grid draws a 500-row sample because nearly a thousand points become a solid mass. Every
statistic quoted above was computed on all 1,020 rows.

## 7. Type-specific analysis

The assignment's adaptation table lists eleven data types. Which apply here decides what
belongs in this section — and saying which do not is part of the answer.

| Row in the assignment's table | Applies? | Reason |
|---|---|---|
| Dates or timestamps | **No** | No capture timestamp in the release; explained below rather than improvised. |
| A target variable | **Yes** | `ui_type` and the geometry features are candidate predictors; balance checked above. |
| Groups or repeated measures | **Yes** | Requests are nested in 17 applications and 6 groups. |
| Free text | **Yes** | 1,020 instructions; length, vocabulary and repetition analysed below. |
| Images | **Yes** | 1,020 PNGs; existence, size, mode, corruption and duplication checked. |
| Audio or signals | No | No audio data. |
| Geographic data | No | No coordinates. |
| Survey or Likert data | No | No respondent-level items; `instruction_cn` is a translation, not a second respondent. |
| More than 30 columns | No | 29 columns after derivation, 9 analysed as numeric. |
| More than 1,000,000 rows | No | 1,020 rows; no sampling of summaries needed. |
| Several linked tables | Partly | 17 annotation files plus an image tree, joined on `img_filename`; verified in section 2. |

### Images

A summary of the image set by application, then a visual check that the boxes land on the
targets. Numbers can show that an annotation is consistent; only an image can show that it is
correct.

In [ ]:
image_table = (clean.groupby("application")
               .agg(screenshots=("img_filename", "nunique"),
                    megapixels_median=("screen_megapixels", "median"),
                    width_min=("img_size_w", "min"),
                    width_max=("img_size_w", "max"),
                    aspect_median=("screen_aspect", "median"))
               .sort_values("screenshots", ascending=False))
image_table["shape"] = image_table["aspect_median"].round(2).map(
    {1.33: "4:3", 1.5: "3:2", 1.6: "16:10", 1.78: "16:9", 3.56: "32:9"}).fillna("other")
display(image_table.round(3))

print("screenshots:", clean["img_filename"].nunique(),
      "| distinct contents:", clean["img_filename"].nunique() - len(digests))
print("colour modes:", audit["mode"].unique().tolist(),
      "| unreadable:", int((audit["status"] != "ok").sum()))

All 1,020 screenshots are RGB PNGs, none corrupt, and the resolution profile is dominated by
16:9 and 32:9 with a few 16:10 and 3:2 captures. Images per application run from 31
(Illustrator) to 93 (MATLAB), matching the request counts exactly because each request has its
own file.

Six crops follow, from three applications with visibly different interface densities, each
with the annotated box outlined. This is the check a reader can perform by eye, and the one
no summary statistic can replace: a box drawn on the wrong widget would invalidate everything
above.

In [ ]:
viz.plot_target_crops(clean, DATA_DIR / "images",
                      ["origin", "windows_common", "solidworks"], n=2, seed=RANDOM_STATE)
plt.show()

In all six crops the red box lands on the requested element: an Origin toolbar icon, a Windows
system entry, a SolidWorks feature-manager row.

The crops also justify the zoom. At full 5120x1440 scale a 16-pixel icon is a few pixels of
ink, so a reviewer looking at the whole screenshot would see nothing to judge; the window is
cropped around the target with roughly a fifth of the screen kept as context.

### Free text

Instructions are short, so the text checks are length, vocabulary and repetition: the three
properties that decide whether an instruction can identify a target on its own.

In [ ]:
STOPWORDS = {"the", "a", "an", "to", "of", "in", "on", "for", "with", "and", "or", "at", "by",
             "from", "this", "that", "it", "is", "as", "into", "all"}
terms = analysis.top_terms(clean["instruction"], STOPWORDS, top_n=20)
display(terms)

viz.plot_top_terms(terms)
plt.show()

The vocabulary is interface verbs and nouns: *open* (71), *select* (66), *change* (58), *tool*
(50), *add* (47), *file* (42), *plot* (41), *edit* (39), *view*, *create*, *insert*, *close*,
*selected*, *color*, *set*.

One of the top terms is a **product name**: `vivado` appears 70 times, second only to *open*.
Many instructions read "open exist project in vivado" or "search report in vivado", so the
application is named inside the text. Part of the
instruction encodes the application, which makes instruction text a weak proxy for the very
variable that section 6 showed to be confounded with resolution.

The instructions also name a target *type* rather than a location. The position check below
measures how far that goes.

In [ ]:
POSITION_TERMS = ["top", "bottom", "left", "right", "upper", "lower", "middle", "center",
                  "centre", "corner", "side", "next to", "above", "below", "near", "inside"]
mentions_position = analysis.position_usage(clean["instruction"], POSITION_TERMS)

print("instructions naming a position:",
      int(mentions_position.sum()),
      f"({mentions_position.mean() * 100:.1f}%)")
display(pd.crosstab(mentions_position.map({True: "mentions position", False: "no position"}),
                    clean["ui_type"], normalize="index").mul(100).round(1))

print()
display(clean.loc[mentions_position, ["instruction", "ui_type", "target_area_pct"]].head(6))

Only **25 instructions (2.5%)** name any position or direction: 13 icon requests (3.3% of the
icon group) and 12 text requests (1.9%). The split is not perfectly even, but at these counts the
difference is a handful of rows either way. The examples are the easy requests — *"Top left
corner"*, *"align the text to the center"* — and 25 out of 1,020 is far too few to distort the
aggregate. The size figures in section 6 stand.

That count needed a correction: a plain substring search returns 37 rows, because it counts
*"centerline"* (contains "center"), *"stop"* (contains "top"), *"Linear"* / *"nearby"* (contain
"near"), *"corners"* (contains "corner") and *"desktop"* (contains "top") as position words.
Matching on word boundaries is what makes the number mean what it says, and it is why the count
is 25 and not 37.

So the text is not doing the disambiguation work: it names what to look for, and the screenshot
decides where. Combined with 25 reused instruction strings across 60 requests, that is the
honest limit of the text: **instruction text alone cannot identify a target**, and it cannot be
used as a join key in a later failure analysis.

### Groups and repeated measures

Rows are nested in applications. The sharper question is whether two rows can describe the same
screen, which section 3 already answered: 14 duplicated pairs. The variance decomposition shows
how much of the size variation lives between applications rather than within them.

In [ ]:
per_screen = clean["img_filename"].value_counts()
print("requests per filename: max =", per_screen.max(),
      "| filenames with more than one request:", int((per_screen > 1).sum()))

display(clean.groupby("group").agg(applications=("application", "nunique"),
                                   requests=("id", "size"))
        .sort_values("requests", ascending=False))

share = features.variance_share(clean, "target_area_pct", "application")
for key, value in share.items():
    print(f"{key:>12}: {value}")

By filename every screen carries exactly one request, so the nesting is only inside
applications. By *content* 14 pairs are the same screen asked twice (section 3), so 28 rows
are dependent. That distinction is easy to miss and is recorded in the limitations.

**16.9% of the variance in log target area sits between applications**, leaving 83.1% within
them. Both numbers are useful: the 83.1% within-application share says application cannot
explain target size, and the 16.9% between-application share is large enough that a model
ignoring application will understate uncertainty on any per-application claim.

### Time

Not applicable, stated explicitly rather than omitted. The annotation files carry **no capture
date, no session identifier and no ordering variable**. Filenames such as
`screenshot_2024-11-06_19-36-34.png` contain an apparent wall-clock reading, but they come from
an unknown number of capture sessions on an unknown number of machines, and the 14 duplicated
pairs sit 9–36 seconds apart — so the stamp is a per-capture clock, not a study timeline.

Building a trend from those strings would be inventing a variable. Answering any time question —
does difficulty drift across a session, are some capture days different — needs the capture
metadata from the pipeline that produced the images, which this release does not include.

## 8. Summary of findings

1. The corpus holds **1,020 grounding requests across 17 applications and 6 industry groups**,
   verified losslessly against the 17 raw annotation files (difference 0 in every file). Those
   requests describe **992 distinct screen contents**: 14 pairs of screenshots are byte-identical,
   captured 9–36 seconds apart, each carrying two different instructions.
2. **MATLAB is a Windows capture mislabelled `macos`** in the release. Correcting the label is what
   makes the corpus 1,020 rather than 927, and the evidence is the captures themselves.
3. The median annotated target covers **0.0382% of its screen** (IQR 0.014–0.081), about 791 px² on
   a 1920x1080 display, with a range from 0.002% to 4.73%.
4. Icon targets are **4.7x smaller in median relative area than text targets** (0.0144% vs 0.0684%,
   Mann-Whitney U = 29,449, p = 7.2e-94, rank-biserial +0.76), the strongest relationship in the data.
5. Median target size varies **11.8x across applications**, from 0.0067% (Origin) to 0.0783%
   (MATLAB), partly interface style and partly target-type mix.
6. Target-type mix differs sharply by application (**chi-square = 192.9, dof = 16, p = 2.2e-32**;
   Stata 83.7% icon against EViews 14.0%), so application averages are not comparable at face value.
   **16.9% of the variance in log target area is between applications**, 83.1% within them.
7. Screen resolution **cannot be separated from application** here: 15 of 17 applications were
   captured at a single resolution, and the pooled association is weak (rho = -0.088, p = 0.005).
8. Instructions are short — median **4 words, 20 characters**, 7.6% single-word — and length is
   unrelated to target size (rho = -0.023, p = 0.46) and to target type (p = 0.45).
9. Only **2.5% of instructions name a position** (25 requests), and 25 instruction strings are
   reused across 60 requests, so the text cannot disambiguate a target on its own.
10. Data quality is high but not perfect: **no missing values, no duplicate rows or ids, no corrupt
    files**, one bounding-box coordinate at -1 and 8 instructions with stray whitespace (both
    repaired), 14 duplicated screenshot pairs, and one mislabelled platform.

### Answers to the four questions

**Q1 — Target size.** The median target is 0.0382% of screen area, and the distribution is
log-normal over three orders of magnitude. Icon targets are 4.7x smaller in median area than text
targets, a gap that holds inside nearly every application where both types occur. Target size must
be reported per target type; a single pooled figure averages two distinct populations.

**Q2 — Corpus balance.** 1,020 requests across 17 applications (31–93 each) and 6 groups (45–297
each). Per application the imbalance is tolerable; per group it is severe, with Creative holding
29.1% of the corpus and Dev 4.4%. Group averages are averages over unequal samples, and
application averages are confounded with target-type mix (chi-square = 192.9, p = 2.2e-32).

**Q3 — Resolution and size.** The question **cannot be answered with this data**. Pooled, higher
resolution is associated with a marginally smaller relative target (rho = -0.088), but 15 of 17
applications were captured at exactly one resolution, so there is no within-application comparison
to make and the association is indistinguishable from an application effect. Separating the two
needs the same application captured at several resolutions, a condition this benchmark lacks.

**Q4 — Instruction text.** Instructions are short (median 4 words, 20 characters), stylistically
uniform across target types, and dominated by interface nouns and verbs, with `vivado` appearing
directly in 70 instructions. Length carries no information about the target. Only 25 instructions
(2.5%) mention a position, and 60 requests reuse an instruction string, so the text names *what* to
find but cannot locate it.

**A question this data cannot answer:** whether any of these geometry features *predict* or
*cause* grounding failure. The release contains no model outputs, no success labels and no
confidence scores. Every claim above is about the corpus, not about what a model will do on it.

## 9. Limitations and next steps

### Limitations

- **One target per screenshot.** Only the requested element is annotated, so the unrequested
  elements on the same screen are invisible. Element detection cannot be evaluated from this data.
- **No outcome variable.** There is no grounding result, label or confidence score in the release,
  so nothing here is a statement about model accuracy. The size measurements describe the task's
  difficulty, not any model's performance on it.
- **14 duplicated screens.** 28 requests describe 14 screens that were captured twice. The
  instructions differ, so the rows are individually valid, but they are not independent
  observations. Rows are also nested within 17 applications, which is a small number of clusters.
- **Resolution and application are confounded.** 15 of 17 applications were captured at one
  resolution, so Q3 is unanswerable here. This is the most consequential limitation for any
  modelling that would treat resolution as a feature.
- **No capture metadata.** DPI scaling, display scaling, theme, zoom level and application build at
  capture time are unrecorded, so the application-level differences in section 6 cannot be
  attributed to interface design rather than rendering conditions.
- **Selection bias of a curated benchmark.** The 1,020 requests were chosen by expert annotators to
  be representatively *hard*. Median target size is therefore not an estimate of how small targets
  are in professional software generally, and no prevalence claim is supported.
- **Coarse group labels.** The 6 industry groups are the benchmark's own, and the smallest holds 45
  rows, so group-level averages rest on thin evidence.
- **Target-area outliers were kept** — 41 IQR-flagged rows, inspected and retained as legitimate
  large regions. Analyses therefore use medians and rank statistics, which the skew makes mandatory
  anyway.
- **No time dimension.** The release carries no timestamp, so nothing here addresses drift, session
  effects or recency.
- **The perceptual hash is unusable on GUI screenshots.** It reported 2,554 bit-identical pairs
  where the true duplicate count is 14, and 30 of those collisions cross an application boundary.
  Any later deduplication must use content digests or pixel-level comparison, not an average hash.

### Next steps

- **Attach outcome labels and test the size claim.** The natural continuation of Q1 is whether
  relative target area and edge distance predict failure. That requires labelling requests
  `correct`, `mislocalized`, `not_found` or `ambiguous` — the unit the project's analysis-unit
  decision already fixes.
- **Report size per target type in every later model.** A 4.7x median gap means one coefficient
  across icons and text will mislead; fit them separately or include the interaction.
- **Capture one application at several resolutions.** Only a controlled condition can separate the
  resolution effect from the application effect that Q3 could not untangle. This is the cheapest
  experiment that would add information this benchmark does not have.
- **Treat the 14 duplicated screens explicitly** if requests are used as independent observations,
  grouping them or keeping one per screen for any test that assumes independence.
- **Verify platform from the captures, not the metadata**, in the project's own pipeline. A release
  can label a Windows corpus as macOS in three places at once and still be wrong.
- **Use content digests, not perceptual hashes**, for deduplication in the project's own capture
  pipeline, and re-check whether that pipeline produces the same one-target-per-screen format.
  On this data an average hash collides far more often than it discriminates.
- **Compare against self-collected captures** to test whether this benchmark's size profile
  transfers to the target environment, since the whole point of the project is the latter.

## 10. References

- Li, K., Meng, Z., Lin, H., Luo, Z., Tian, Y., Ma, J., Huang, Z., & Chua, T.-S. (2025).
  *ScreenSpot-Pro: GUI Grounding for Professional High-Resolution Computer Use.*
  arXiv:2504.07981. https://doi.org/10.48550/arXiv.2504.07981
- Dataset: `likaixin/ScreenSpot-Pro`, Hugging Face. https://huggingface.co/datasets/likaixin/ScreenSpot-Pro
  — retrieved 19 November 2026, revision `main`; MIT licence covering the annotations.
- Benchmark page and leaderboard: https://gui-agent.github.io/grounding-leaderboard
- pandas, NumPy, Matplotlib, seaborn, SciPy and Pillow — library documentation, used for the
  analysis and the figures.
- AI assistance: an AI coding assistant (Hermes Agent) was used to draft the analysis modules and
  the download script `src/isdm/data/fetch.py`. Every figure and number in this notebook is
  produced by the executed cells above and was checked against the raw annotation files.

## Appendix: exporting the executed notebook to PDF

This is an export utility, not part of the analysis. Run **Kernel → Restart & Run All**, save the
notebook (`Ctrl+S`), then run the cell below. It writes:

- `runs/eda/EDA_Beira_John-Ronan.pdf` using `nbconvert` → LaTeX → `xelatex`;
- `runs/eda/EDA_Beira_John-Ronan.html` as the assignment's accepted fallback, which needs no LaTeX.

PDF export needs `xelatex` and `pandoc`. This repository's dev dependencies provide pandoc through
`pypandoc-binary`; `xelatex` comes from a TeX distribution. If either is missing, the cell prints
the requirement and still writes the HTML. The notebook itself remains runnable without them.


In [ ]:
# Export the saved notebook to PDF (LaTeX) and HTML (fallback).
# Run All first, save the notebook, then run this cell. This is a utility, not analysis.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile

NOTEBOOK_NAME = "EDA_Beira_John-Ronan.ipynb"

# Locate the notebook whether the kernel starts in the repo root, notebooks/, or Colab.
search = [Path.cwd(), *Path.cwd().parents]
notebook = next(
    (p for p in (root / "notebooks" / NOTEBOOK_NAME for root in search) if p.is_file()),
    None,
) or next(
    (root / NOTEBOOK_NAME for root in search if (root / NOTEBOOK_NAME).is_file()),
    None,
)
if notebook is None:
    raise FileNotFoundError(f"Could not find {NOTEBOOK_NAME}")

repo_root = notebook.parent.parent if notebook.parent.name == "notebooks" else notebook.parent
out_dir = repo_root / "runs" / "eda"
out_dir.mkdir(parents=True, exist_ok=True)

saved = json.loads(notebook.read_text())
if not any(cell.get("outputs") for cell in saved["cells"] if cell.get("cell_type") == "code"):
    print("WARNING: the saved notebook has no outputs. Run All and save before exporting.")

python = sys.executable
env = os.environ.copy()

# nbconvert's LaTeX path needs pandoc. Prefer one on PATH, else pypandoc's bundled binary.
pandoc = shutil.which("pandoc")
if pandoc is None:
    try:
        import pypandoc
        pandoc = pypandoc.get_pandoc_path()
    except Exception:
        pandoc = None
if pandoc:
    env["PATH"] = str(Path(pandoc).parent) + os.pathsep + env.get("PATH", "")

def run(command, cwd=None):
    return subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)

# HTML always works and is the assignment's accepted fallback.
html = run([python, "-m", "nbconvert", "--to", "html",
            "--output-dir", str(out_dir), "--output", "EDA_Beira_John-Ronan.html", str(notebook)])
html_path = out_dir / "EDA_Beira_John-Ronan.html"
print("HTML:", html_path if html.returncode == 0 else html.stderr[-1200:])

xelatex = shutil.which("xelatex", path=env["PATH"])
if pandoc is None or xelatex is None:
    print(
        "PDF skipped: install pandoc and xelatex, then re-run. "
        "The HTML export above is complete."
    )
else:
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        latex = run([python, "-m", "nbconvert", "--to", "latex",
                     "--output-dir", str(tmp), "--output", "notebook.tex", str(notebook)])
        tex = tmp / "notebook.tex"
        if latex.returncode or not tex.is_file():
            print("LaTeX export failed:", latex.stderr[-1200:])
        else:
            # pandoc 3.x emits \LTcaptype{none}; the nbconvert template needs this counter.
            source = tex.read_text()
            if r"\newcounter{none}" not in source:
                source = source.replace(
                    r"\begin{document}", r"\newcounter{none}" + "\n" + r"\begin{document}", 1
                )
                tex.write_text(source)
            for _ in range(2):
                built = run(
                    [xelatex, "-interaction=nonstopmode", "-halt-on-error", tex.name],
                    cwd=tmp,
                )
            pdf = tmp / "notebook.pdf"
            if built.returncode == 0 and pdf.is_file():
                target = out_dir / "EDA_Beira_John-Ronan.pdf"
                shutil.copyfile(pdf, target)
                print("PDF:", target)
            else:
                print("xelatex failed:", built.stderr[-1200:])
